In [ ]:
#@title 🧰 A. SETUP AND CONFIGURATION

This notebook implements the complete Google Earth Engine Python workflow used for satellite–field matchup generation, spectral feature engineering and selection, machine-learning model development and evaluation, and spatio-temporal prediction of Secchi disk depth (SDD) and chlorophyll-a (Chl-a).

The workflow is designed for execution in Google Colab. Users require a Google Earth Engine account and must authenticate the Earth Engine API using their own account. Google Drive is used for reading input datasets and storing intermediate and final outputs. User-specific paths and analysis settings should be defined in the configuration sections before running the workflow.

In [ ]:
#@title 🔗 A1. Mount Google Drive
# Mount Google Drive to access input data and save workflow outputs
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#@title 🔑 A2. Install Required Packages
!pip -q install \
    earthengine-api \
    numpy \
    pandas \
    matplotlib \
    seaborn \
    openpyxl \
    rasterio \
    geopandas \
    scipy \
    scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 46.2 MB/s eta 0:00:00
  Created wheel for simplekml: filename=simplekml-1.3.6-py3-none-any.whl size=65860 sha256=e8c5ae05527f92ccab83df9fce9075cadb1789ea122c4012faa827fdb46d2f20
  Stored in directory: /root/.cache/pip/wheels/83/ee/f2/65cecfd948f1429ead035fd6d56bc6bd6574a636ddc4d65cbd
Successfully built simplekml


In [ ]:
#@title 📦 A3. Import Libraries and Functions

import ee
import os
import re
import glob
import warnings
import rasterio
import numpy as np
import pandas as pd
import seaborn as sns
import geopandas as gpd
import matplotlib.pyplot as plt

from scipy.stats import randint
from itertools import combinations, permutations
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.feature_selection import VarianceThreshold

warnings.filterwarnings("ignore")

In [ ]:
#@title 🔑 A4. Authenticate and Initialize Google Earth Engine

# Enter your Google Cloud project ID associated with Google Earth Engine
GEE_PROJECT = "your-google-cloud-project-id"  #@param {type:"string"}

# Authenticate the current user and initialize Earth Engine
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)

print("Google Earth Engine initialized successfully.")

In [ ]:
#@title 📁 A5. Configure Input and Output Directories

# Google Drive root mounted by A1
DRIVE_ROOT = "/content/drive/MyDrive"  #@param {type:"string"}

# Main directory for this workflow
WORKFLOW_FOLDER = "GEE_Water_Quality_Pipeline"  #@param {type:"string"}
BASE_DIR = os.path.join(DRIVE_ROOT, WORKFLOW_FOLDER)

# Input and output directories
INPUT_DIR = os.path.join(BASE_DIR, "input")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

# Create output directory if it does not already exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Input directory:  {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
#@title 🚤 1. IN-SITU DATA PREPARATION AND EXPLORATION

In-situ water-quality observations were obtained from the New York State Citizens Statewide Lake Assessment Program (CSLAP). The original monitoring data are provided in long format, with individual water-quality parameters stored as separate observations.

This section prepares the field observations for subsequent satellite–field matchup and modelling. The workflow standardizes column names and dates, retains the required water-quality parameters and associated metadata, and restructures the observations so that parameters collected at the same lake, site, and date are stored in a single record. The resulting dataset is used for Landsat spectral extraction and subsequent machine-learning analysis.

Exploratory summaries are also generated to examine the spatial and temporal distribution of the available observations and the distributions of Secchi disk depth (SDD) and chlorophyll-a (Chl-a).

In [ ]:
#@title 1.1. Define In-situ Data and Parameters

# ---------------------------------------------------
# INPUT AND OUTPUT DIRECTORIES
# ---------------------------------------------------

# Folder containing the raw CSLAP Excel files
IN_SITU_RAW_DIR = os.path.join(INPUT_DIR, "CSLAP_raw")

# Folder for processed in-situ outputs
IN_SITU_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "in_situ")
os.makedirs(IN_SITU_OUTPUT_DIR, exist_ok=True)

# Find all Excel files in the raw-data directory
files = glob.glob(os.path.join(IN_SITU_RAW_DIR, "*.xlsx"))

# ---------------------------------------------------
# LAKE ORDER
# ---------------------------------------------------

LAKE_ORDER = [
    "Otisco",
    "Skaneateles",
    "Owasco",
    "Cayuga",
    "Seneca",
    "Keuka",
    "Canandaigua",
    "Honeoye",
    "Canadice",
    "Hemlock",
    "Conesus"
]

# ---------------------------------------------------
# WATER-QUALITY PARAMETERS
# ---------------------------------------------------

PARAMETER_MAP = {
    "chlorophyll-a": "chl_a",
    "secchi_disk_depth": "sdd",
    "apparent_color": "apparent_color",
    "phosphorus": "phosphorus",
    "temperature": "temperature",
    "turbidity": "turbidity",
    "wind_current_condition": "wind_current_condition"
}

# Columns used to uniquely identify each field observation
BASE_COLUMNS = [
    "WATERBODY_NAME",
    "EVENT_DATETIME",
    "SITE_CODE",
    "LATITUDE",
    "LONGITUDE"
]

# Dictionary for storing processed data for each lake
lake_results = {}

print(f"CSLAP files found: {len(files)}")

for file in files:
    print(" -", os.path.basename(file))

In [ ]:
#@title 1.2. Clean and Restructure In-situ Observations

# ---------------------------------------------------
# PROCESS EACH CSLAP FILE
# ---------------------------------------------------

for file in files:

    print(f"\nProcessing: {os.path.basename(file)}")

    df = pd.read_excel(file)

    # Standardize column names
    df.columns = (
        df.columns
        .str.strip()
        .str.replace(" ", "_", regex=False)
    )

    # Remove duplicated columns, if present
    df = df.loc[:, ~df.columns.duplicated()]

    # ---------------------------------------------------
    # CHECK REQUIRED COLUMNS
    # ---------------------------------------------------

    required_cols = BASE_COLUMNS + [
        "SAMPLE_LOCATION",
        "SAMPLE_ORGANIZATION",
        "SAMPLE_DEPTH_METERS",
        "SAMPLE_METHOD",
        "PARAMETER_NAME",
        "RESULT_VALUE",
        "RESULT_CATEGORY",
        "UNIT",
        "REPORTING_DETECTION_LIMIT"
    ]

    missing_cols = [
        column for column in required_cols
        if column not in df.columns
    ]

    if missing_cols:
        print(f"Skipped: missing columns {missing_cols}")
        continue

    # ---------------------------------------------------
    # PREPARE RESULT VALUES
    # ---------------------------------------------------

    # Where RESULT_VALUE is missing, retain a valid categorical
    # result when available.
    df["RESULT_VALUE"] = df.apply(
        lambda row:
            row["RESULT_CATEGORY"]
            if (
                pd.isna(row["RESULT_VALUE"])
                and pd.notna(row["RESULT_CATEGORY"])
                and str(row["RESULT_CATEGORY"]).lower()
                != "not_applicable"
            )
            else row["RESULT_VALUE"],
        axis=1
    )

    # Retain only required fields
    df = df[required_cols]

    # Convert observation date to datetime
    df["EVENT_DATETIME"] = pd.to_datetime(
        df["EVENT_DATETIME"],
        errors="coerce"
    )

    # Remove records without a valid observation date
    df = df.dropna(subset=["EVENT_DATETIME"])

    # ---------------------------------------------------
    # FILTER WATER-QUALITY PARAMETERS
    # ---------------------------------------------------

    df = df[
        df["PARAMETER_NAME"].isin(PARAMETER_MAP.keys())
    ].copy()

    df["PARAMETER_NAME"] = (
        df["PARAMETER_NAME"]
        .map(PARAMETER_MAP)
    )

    if df.empty:
        print("Skipped: no selected parameters found.")
        continue

    # Determine lake name
    lake_name = (
        str(df["WATERBODY_NAME"].iloc[0])
        .replace("Lake", "")
        .strip()
    )

    # ---------------------------------------------------
    # RESTRUCTURE LONG-FORM DATA
    # ---------------------------------------------------

    value_df = df.pivot_table(
        index=BASE_COLUMNS,
        columns="PARAMETER_NAME",
        values="RESULT_VALUE",
        aggfunc="first"
    )

    unit_df = df.pivot_table(
        index=BASE_COLUMNS,
        columns="PARAMETER_NAME",
        values="UNIT",
        aggfunc="first"
    ).add_suffix("_unit")

    detection_df = df.pivot_table(
        index=BASE_COLUMNS,
        columns="PARAMETER_NAME",
        values="REPORTING_DETECTION_LIMIT",
        aggfunc="first"
    ).add_suffix("_det_limit")

    method_df = df.pivot_table(
        index=BASE_COLUMNS,
        columns="PARAMETER_NAME",
        values="SAMPLE_METHOD",
        aggfunc="first"
    ).add_suffix("_method")

    location_df = df.pivot_table(
        index=BASE_COLUMNS,
        columns="PARAMETER_NAME",
        values="SAMPLE_LOCATION",
        aggfunc="first"
    ).add_suffix("_location")

    organization_df = df.pivot_table(
        index=BASE_COLUMNS,
        columns="PARAMETER_NAME",
        values="SAMPLE_ORGANIZATION",
        aggfunc="first"
    ).add_suffix("_org")

    depth_df = df.pivot_table(
        index=BASE_COLUMNS,
        columns="PARAMETER_NAME",
        values="SAMPLE_DEPTH_METERS",
        aggfunc="first"
    ).add_suffix("_depth")

    # Combine parameter values and associated metadata
    final_df = pd.concat(
        [
            value_df,
            unit_df,
            detection_df,
            method_df,
            location_df,
            organization_df,
            depth_df
        ],
        axis=1
    ).reset_index()

    lake_results[lake_name] = final_df

    print(
        f"{lake_name}: "
        f"{len(df)} source records → "
        f"{len(final_df)} observation records"
    )


# ---------------------------------------------------
# EXPORT PROCESSED DATA
# ---------------------------------------------------

IN_SITU_FILE = os.path.join(
    IN_SITU_OUTPUT_DIR,
    "CSLAP_InSitu_2017-2025.xlsx"
)

with pd.ExcelWriter(
    IN_SITU_FILE,
    engine="openpyxl"
) as writer:

    written_sheets = 0

    for lake in LAKE_ORDER:

        if lake not in lake_results:
            print(f"No processed data found for {lake}")
            continue

        lake_results[lake].to_excel(
            writer,
            sheet_name=lake[:31],
            index=False
        )

        written_sheets += 1

if written_sheets == 0:
    raise ValueError(
        "No CSLAP datasets were processed. "
        "Check the input files and directory."
    )

print(
    f"\nProcessed in-situ dataset saved to:\n"
    f"{IN_SITU_FILE}"
)

In [ ]:
#@title 1.3. Generate Lake-wise Summary of In-situ Observations

# ---------------------------------------------------
# LOAD PROCESSED IN-SITU DATA
# ---------------------------------------------------

excel_file = pd.ExcelFile(IN_SITU_FILE)

summary_records = []

for lake in excel_file.sheet_names:

    df = pd.read_excel(
        IN_SITU_FILE,
        sheet_name=lake
    )

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
    )

    # Number of available observations
    sdd_count = (
        df["sdd"].notna().sum()
        if "sdd" in df.columns
        else 0
    )

    chla_count = (
        df["chl_a"].notna().sum()
        if "chl_a" in df.columns
        else 0
    )

    # Number of observations where both variables are available
    paired_count = 0

    if {"sdd", "chl_a"}.issubset(df.columns):
        paired_count = (
            df[["sdd", "chl_a"]]
            .dropna()
            .shape[0]
        )

    summary_records.append({
        "Lake": lake,
        "SDD_Count": sdd_count,
        "Chl_a_Count": chla_count,
        "SDD_Chl_a_Count": paired_count
    })

lake_summary_df = pd.DataFrame(summary_records)

# Apply study lake order
lake_summary_df["Lake"] = pd.Categorical(
    lake_summary_df["Lake"],
    categories=LAKE_ORDER,
    ordered=True
)

lake_summary_df = (
    lake_summary_df
    .sort_values("Lake")
    .reset_index(drop=True)
)

# Display summary
display(lake_summary_df)

# ---------------------------------------------------
# SAVE SUMMARY
# ---------------------------------------------------

LAKE_SUMMARY_FILE = os.path.join(
    IN_SITU_OUTPUT_DIR,
    "CSLAP_Lakewise_Summary.xlsx"
)

lake_summary_df.to_excel(
    LAKE_SUMMARY_FILE,
    index=False
)

print(
    f"Lake-wise summary saved to:\n"
    f"{LAKE_SUMMARY_FILE}"
)

In [ ]:
#@title 1.4. Generate Monthly Summary of In-situ Observations

# ---------------------------------------------------
# SETTINGS
# ---------------------------------------------------

VARIABLES = {
    "sdd": "SDD",
    "chl_a": "Chl_a"
}

YEARS = range(2017, 2026)

monthly_results = []

# ---------------------------------------------------
# PROCESS EACH LAKE
# ---------------------------------------------------

excel_file = pd.ExcelFile(IN_SITU_FILE)

for lake in excel_file.sheet_names:

    df = pd.read_excel(
        IN_SITU_FILE,
        sheet_name=lake
    )

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
    )

    df["event_datetime"] = pd.to_datetime(
        df["event_datetime"],
        errors="coerce"
    )

    df = df.dropna(
        subset=["event_datetime"]
    )

    df["Year"] = (
        df["event_datetime"]
        .dt.year
    )

    df["Month"] = (
        df["event_datetime"]
        .dt.month
    )

    # Restrict to study period
    df = df[
        df["Year"].isin(YEARS)
    ]

    for variable, variable_name in VARIABLES.items():

        if variable not in df.columns:
            continue

        grouped = (
            df.groupby(
                ["Year", "Month"]
            )[variable]
            .agg(
                Mean="mean",
                Count="count"
            )
            .reset_index()
        )

        grouped["Lake"] = lake
        grouped["Variable"] = variable_name

        monthly_results.append(grouped)

# ---------------------------------------------------
# COMBINE RESULTS
# ---------------------------------------------------

monthly_summary_df = pd.concat(
    monthly_results,
    ignore_index=True
)

monthly_summary_df["Date"] = pd.to_datetime(
    dict(
        year=monthly_summary_df["Year"],
        month=monthly_summary_df["Month"],
        day=1
    )
)

monthly_summary_df = monthly_summary_df[
    [
        "Lake",
        "Variable",
        "Date",
        "Mean",
        "Count"
    ]
]

monthly_summary_df = (
    monthly_summary_df
    .sort_values(
        ["Lake", "Variable", "Date"]
    )
    .reset_index(drop=True)
)

display(monthly_summary_df.head())

# ---------------------------------------------------
# SAVE SUMMARY
# ---------------------------------------------------

MONTHLY_SUMMARY_FILE = os.path.join(
    IN_SITU_OUTPUT_DIR,
    "CSLAP_Monthly_Summary.xlsx"
)

monthly_summary_df.to_excel(
    MONTHLY_SUMMARY_FILE,
    index=False
)

print(
    f"Monthly summary saved to:\n"
    f"{MONTHLY_SUMMARY_FILE}"
)

In [ ]:
#@title 1.5. Examine SDD and Chl-a Distributions and Transformations

# ---------------------------------------------------
# LOAD AND COMBINE ALL LAKES
# ---------------------------------------------------

all_sheets = pd.read_excel(
    IN_SITU_FILE,
    sheet_name=None
)

dataframes = []

for lake, df in all_sheets.items():

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
    )

    if {"sdd", "chl_a"}.issubset(df.columns):

        temp_df = df[
            ["sdd", "chl_a"]
        ].copy()

        temp_df["Lake"] = lake

        dataframes.append(temp_df)

combined_df = pd.concat(
    dataframes,
    ignore_index=True
)

# ---------------------------------------------------
# CLEAN TARGET VARIABLES
# ---------------------------------------------------

combined_df["sdd"] = pd.to_numeric(
    combined_df["sdd"],
    errors="coerce"
)

combined_df["chl_a"] = pd.to_numeric(
    combined_df["chl_a"],
    errors="coerce"
)

combined_df = combined_df.replace(
    [np.inf, -np.inf],
    np.nan
)

combined_df = combined_df.dropna(
    subset=["sdd", "chl_a"]
)

# Log transformation requires positive values
combined_df = combined_df[
    (combined_df["sdd"] > 0)
    & (combined_df["chl_a"] > 0)
].copy()

# ---------------------------------------------------
# LOG10 TRANSFORMATIONS
# ---------------------------------------------------

combined_df["log_sdd"] = np.log10(
    combined_df["sdd"]
)

combined_df["log_chl_a"] = np.log10(
    combined_df["chl_a"]
)

# ---------------------------------------------------
# PLOT DISTRIBUTIONS
# ---------------------------------------------------

plot_variables = [
    ("sdd", "Observed SDD (m)"),
    ("log_sdd", "Log10 SDD"),
    ("chl_a", "Observed Chl-a (µg/L)"),
    ("log_chl_a", "Log10 Chl-a")
]

fig, axes = plt.subplots(
    2,
    2,
    figsize=(14, 10)
)

axes = axes.flatten()

for ax, (variable, title) in zip(
    axes,
    plot_variables
):

    data = combined_df[
        variable
    ].dropna()

    ax.hist(
        data,
        bins=20,
        edgecolor="black",
        rwidth=0.85
    )

    ax.set_title(
        title,
        fontsize=14
    )

    ax.set_xlabel(
        "Observed value",
        fontsize=12
    )

    ax.set_ylabel(
        "Frequency",
        fontsize=12
    )

    ax.grid(
        alpha=0.3
    )

    stats_text = (
        f"N = {len(data)}\n"
        f"Mean = {data.mean():.2f}\n"
        f"Median = {data.median():.2f}\n"
        f"SD = {data.std():.2f}\n"
        f"Min = {data.min():.2f}\n"
        f"Max = {data.max():.2f}"
    )

    ax.text(
        0.97,
        0.95,
        stats_text,
        transform=ax.transAxes,
        verticalalignment="top",
        horizontalalignment="right",
        fontsize=11,
        bbox=dict(
            facecolor="white",
            alpha=0.85,
            edgecolor="gray"
        )
    )

plt.tight_layout()

# ---------------------------------------------------
# SAVE FIGURE
# ---------------------------------------------------

distribution_figure = os.path.join(
    IN_SITU_OUTPUT_DIR,
    "CSLAP_SDD_Chla_Distributions.png"
)

plt.savefig(
    distribution_figure,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    f"Distribution figure saved to:\n"
    f"{distribution_figure}"
)

In [ ]:
#@title 🛰️ 2. LANDSAT SPECTRAL EXTRACTION

This section retrieves Landsat 8 and Landsat 9 imagery corresponding to the spatial and temporal extent of the field observations. Top-of-Atmosphere (TOA) and Level-2 Surface Reflectance (SR) products are processed separately to allow comparison of the two Landsat products.

Cloud and cloud-shadow pixels are masked using the Landsat QA_PIXEL band. Spectral information is extracted around each field sampling location using square 3×3, 5×5, and 7×7 pixel kernels. For each spectral band and spatial window, the mean, median, standard deviation, and coefficient of variation (CV) are calculated. These extracted spectral datasets are subsequently used for satellite–field temporal matchup and machine-learning analysis.

In [ ]:
#@title 2.1. Configuration and User Settings
# ---------------------------------------------------
# STUDY PERIOD
# ---------------------------------------------------

START_YEAR = 2017
END_YEAR = 2025

# Day-of-year limits for image retrieval.
# Default values correspond approximately to May 15–October 15.
START_DOY = 135
END_DOY = 288

# ---------------------------------------------------
# LANDSAT WRS-2 PATH / ROW
# ---------------------------------------------------

WRS_PATHS = [15, 16, 17]
WRS_ROW = 30

# ---------------------------------------------------
# LANDSAT PRODUCT
# ---------------------------------------------------

# Choose "TOA" or "SR"
DATA_PRODUCT = "TOA"

# Extraction code iterates over the selected product.
DATA_PRODUCTS = [DATA_PRODUCT]

# ---------------------------------------------------
# SPATIAL EXTRACTION WINDOWS
# ---------------------------------------------------

WINDOW_SIZES = [3, 5, 7]

# ---------------------------------------------------
# OPTIONAL PIXEL FILTERING
# ---------------------------------------------------

# Remove negative reflectance values before calculating
# neighborhood statistics
REMOVE_NEGATIVE_VALUES = True

# Apply median ± (STD_THRESHOLD × standard deviation)
# filtering within each spatial window
APPLY_PIXEL_FILTER = True

# User-defined standard-deviation threshold
STD_THRESHOLD = 1.5

# ---------------------------------------------------
# DISPLAY SETTINGS
# ---------------------------------------------------

print(f"Study years: {START_YEAR}–{END_YEAR}")
print(f"Day-of-year range: {START_DOY}–{END_DOY}")
print(f"Landsat product: {DATA_PRODUCT}")
print(f"WRS paths: {WRS_PATHS}, row: {WRS_ROW}")
print(f"Kernel sizes: {WINDOW_SIZES}")

if APPLY_PIXEL_FILTER:
    print(
        f"Pixel filtering: median ± {STD_THRESHOLD} SD"
    )
else:
    print("Pixel filtering: disabled")

print(
    "Negative-value removal:",
    "enabled" if REMOVE_NEGATIVE_VALUES else "disabled"
)

In [ ]:
#@title 2.2. Load Field Sampling Locations

# ===================================================
# READ PROCESSED IN-SITU DATA
# ===================================================

all_sheets = pd.read_excel(
    IN_SITU_FILE,
    sheet_name=None
)

location_records = []

for lake, df in all_sheets.items():

    # Standardize column names
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
    )

    required_columns = {
        "site_code",
        "latitude",
        "longitude"
    }

    if not required_columns.issubset(df.columns):
        print(
            f"Skipping {lake}: "
            "required coordinate columns not found."
        )
        continue

    # Retain unique sampling locations
    locations = (
        df[
            ["site_code", "latitude", "longitude"]
        ]
        .dropna()
        .drop_duplicates()
        .copy()
    )

    locations["lake"] = lake

    location_records.append(locations)


if not location_records:
    raise ValueError(
        "No valid sampling locations were found "
        "in the processed in-situ dataset."
    )


points_df = pd.concat(
    location_records,
    ignore_index=True
)


# ===================================================
# CONVERT TO EARTH ENGINE FEATURE COLLECTION
# ===================================================

features = []

for _, row in points_df.iterrows():

    longitude = float(row["longitude"])
    latitude = float(row["latitude"])

    feature = ee.Feature(
        ee.Geometry.Point(
            [longitude, latitude]
        ),
        {
            "site_code": str(row["site_code"]),
            "lake": str(row["lake"]),
            "latitude": latitude,
            "longitude": longitude
        }
    )

    features.append(feature)


points_fc = ee.FeatureCollection(features)


# ===================================================
# SUMMARY
# ===================================================

print(
    f"Unique sampling locations: "
    f"{len(points_df)}"
)

print(
    f"Lakes represented: "
    f"{points_df['lake'].nunique()}"
)

In [ ]:
#@title 2.3. Prepare Landsat 8 and 9 Collections

# ===================================================
# LANDSAT BAND DEFINITIONS
# ===================================================

TOA_BANDS = [
    "B1", "B2", "B3", "B4",
    "B5", "B6", "B7"
]

SR_BANDS = [
    "SR_B1", "SR_B2", "SR_B3", "SR_B4",
    "SR_B5", "SR_B6", "SR_B7"
]


# ===================================================
# TOA CLOUD / SHADOW MASK
# ===================================================

def prepare_toa(image):
    """
    Mask cloud and cloud-shadow pixels in Landsat
    Collection 2 Top-of-Atmosphere imagery.
    """

    qa = image.select("QA_PIXEL")

    cloud_mask = qa.bitwiseAnd(1 << 3).eq(0)
    shadow_mask = qa.bitwiseAnd(1 << 4).eq(0)

    clear_mask = cloud_mask.And(shadow_mask)

    return (
        image
        .select(TOA_BANDS)
        .updateMask(clear_mask)
        .copyProperties(
            image,
            image.propertyNames()
        )
    )


# ===================================================
# SURFACE REFLECTANCE PREPARATION
# ===================================================

def prepare_sr(image):
    """
    Mask cloud and cloud-shadow pixels and apply the
    Landsat Collection 2 Level-2 optical-band scale
    factor and offset.
    """

    qa = image.select("QA_PIXEL")

    cloud_mask = qa.bitwiseAnd(1 << 3).eq(0)
    shadow_mask = qa.bitwiseAnd(1 << 4).eq(0)

    clear_mask = cloud_mask.And(shadow_mask)

    # Landsat Collection 2 Level-2 scale and offset
    scaled = (
        image
        .select(SR_BANDS)
        .multiply(0.0000275)
        .add(-0.2)
    )

    return (
        scaled
        .updateMask(clear_mask)
        .copyProperties(
            image,
            image.propertyNames()
        )
    )


# ===================================================
# BUILD LANDSAT 8 + LANDSAT 9 COLLECTION
# ===================================================

def get_landsat_collection(
    landsat8_dataset,
    landsat9_dataset,
    processor
):
    """
    Retrieve and merge Landsat 8 and Landsat 9
    imagery using the user-defined temporal and
    WRS-2 spatial filters.
    """

    landsat8 = (
        ee.ImageCollection(landsat8_dataset)
        .filter(
            ee.Filter.calendarRange(
                START_YEAR,
                END_YEAR,
                "year"
            )
        )
        .filter(
            ee.Filter.dayOfYear(
                START_DOY,
                END_DOY
            )
        )
        .filter(
            ee.Filter.inList(
                "WRS_PATH",
                WRS_PATHS
            )
        )
        .filter(
            ee.Filter.eq(
                "WRS_ROW",
                WRS_ROW
            )
        )
        .map(processor)
    )

    # Landsat 9 imagery begins in 2021.
    landsat9_start = max(
        2021,
        START_YEAR
    )

    landsat9 = (
        ee.ImageCollection(landsat9_dataset)
        .filter(
            ee.Filter.calendarRange(
                landsat9_start,
                END_YEAR,
                "year"
            )
        )
        .filter(
            ee.Filter.dayOfYear(
                START_DOY,
                END_DOY
            )
        )
        .filter(
            ee.Filter.inList(
                "WRS_PATH",
                WRS_PATHS
            )
        )
        .filter(
            ee.Filter.eq(
                "WRS_ROW",
                WRS_ROW
            )
        )
        .map(processor)
    )

    return landsat8.merge(
        landsat9
    )


# ===================================================
# CREATE REQUESTED COLLECTIONS
# ===================================================

landsat_collections = {}

if "TOA" in DATA_PRODUCTS:

    landsat_collections["TOA"] = (
        get_landsat_collection(
            "LANDSAT/LC08/C02/T1_TOA",
            "LANDSAT/LC09/C02/T1_TOA",
            prepare_toa
        )
    )

    print(
        "TOA images:",
        landsat_collections["TOA"]
        .size()
        .getInfo()
    )


if "SR" in DATA_PRODUCTS:

    landsat_collections["SR"] = (
        get_landsat_collection(
            "LANDSAT/LC08/C02/T1_L2",
            "LANDSAT/LC09/C02/T1_L2",
            prepare_sr
        )
    )

    print(
        "SR images:",
        landsat_collections["SR"]
        .size()
        .getInfo()
    )

In [ ]:
#@title 2.4. Convert Earth Engine Features to DataFrame

def fc_to_df(feature_collection):
    """
    Convert an Earth Engine FeatureCollection
    to a pandas DataFrame.

    Parameters
    ----------
    feature_collection : ee.FeatureCollection
        Earth Engine FeatureCollection containing
        extracted spectral statistics.

    Returns
    -------
    pandas.DataFrame
        Feature properties converted to tabular form.
    """

    data = feature_collection.getInfo()

    features = data.get(
        "features",
        []
    )

    if not features:
        return pd.DataFrame()

    rows = [
        feature["properties"]
        for feature in features
    ]

    return pd.DataFrame(rows)

In [ ]:
#@title 2.5. Define Kernel-based Spectral Statistics

def get_square_kernel(window_size):
    """
    Create a square Earth Engine kernel from an
    odd-numbered window size such as 3, 5, or 7.
    """

    if window_size < 3 or window_size % 2 == 0:
        raise ValueError(
            "Window size must be an odd integer "
            "greater than or equal to 3."
        )

    radius = (window_size - 1) // 2

    return ee.Kernel.square(
        radius=radius,
        units="pixels",
        normalize=False
    )


def filter_spectral_pixels(
    image,
    band_names,
    window_size
):
    """
    Optionally remove negative reflectance values
    and local spectral outliers before calculating
    neighborhood statistics.

    Outliers are defined relative to the local
    neighborhood as:

        median ± (STD_THRESHOLD × standard deviation)
    """

    kernel = get_square_kernel(
        window_size
    )

    filtered_bands = []

    for band_name in band_names:

        band = image.select(
            band_name
        )

        # Start with the original band.
        filtered_band = band

        # -------------------------------------------
        # OPTIONAL NEGATIVE-VALUE MASK
        # -------------------------------------------

        if REMOVE_NEGATIVE_VALUES:

            filtered_band = (
                filtered_band
                .updateMask(
                    band.gte(0)
                )
            )

        # -------------------------------------------
        # OPTIONAL LOCAL OUTLIER FILTER
        # -------------------------------------------

        if APPLY_PIXEL_FILTER:

            local_stats = (
                band
                .reduceNeighborhood(
                    reducer=(
                        ee.Reducer.median()
                        .combine(
                            ee.Reducer.stdDev(),
                            sharedInputs=True
                        )
                    ),
                    kernel=kernel
                )
            )

            # Select by output position so the
            # function is independent of generated
            # Earth Engine band-name suffixes.
            local_median = (
                local_stats
                .select([0])
            )

            local_std = (
                local_stats
                .select([1])
            )

            lower_limit = (
                local_median
                .subtract(
                    local_std.multiply(
                        STD_THRESHOLD
                    )
                )
            )

            upper_limit = (
                local_median
                .add(
                    local_std.multiply(
                        STD_THRESHOLD
                    )
                )
            )

            outlier_mask = (
                band
                .gte(lower_limit)
                .And(
                    band.lte(
                        upper_limit
                    )
                )
            )

            filtered_band = (
                filtered_band
                .updateMask(
                    outlier_mask
                )
            )

        filtered_bands.append(
            filtered_band.rename(
                band_name
            )
        )

    return ee.Image.cat(
        filtered_bands
    )


def calculate_neighborhood_statistics(
    image,
    band_names,
    window_size
):
    """
    Calculate mean, median, and standard deviation
    within a square spatial kernel.
    """

    kernel = get_square_kernel(
        window_size
    )

    # Apply optional filtering.
    if (
        REMOVE_NEGATIVE_VALUES
        or APPLY_PIXEL_FILTER
    ):

        spectral_image = (
            filter_spectral_pixels(
                image,
                band_names,
                window_size
            )
        )

    else:

        spectral_image = (
            image.select(
                band_names
            )
        )

    reducer = (
        ee.Reducer.mean()
        .combine(
            ee.Reducer.median(),
            sharedInputs=True
        )
        .combine(
            ee.Reducer.stdDev(),
            sharedInputs=True
        )
    )

    statistics_image = (
        spectral_image
        .reduceNeighborhood(
            reducer=reducer,
            kernel=kernel
        )
    )

    return statistics_image

In [ ]:
#@title 2.6. Extract Spectral Statistics

def extract_spectral_statistics(
    collection,
    band_names,
    window_size,
    year,
    sampling_points
):
    """
    Extract neighborhood spectral statistics for
    every Landsat image and field sampling location
    for a specified year.
    """

    # -----------------------------------------------
    # FILTER COLLECTION TO YEAR
    # -----------------------------------------------

    collection_year = (
        collection
        .filter(
            ee.Filter.calendarRange(
                year,
                year,
                "year"
            )
        )
    )

    # -----------------------------------------------
    # PROCESS EACH LANDSAT IMAGE
    # -----------------------------------------------

    def process_image(image):

        acquisition_date = (
            ee.Date(
                image.get(
                    "system:time_start"
                )
            )
            .format(
                "YYYY-MM-dd"
            )
        )

        image_id = image.get(
            "LANDSAT_PRODUCT_ID"
        )

        # Calculate neighborhood statistics.
        stats_image = (
            calculate_neighborhood_statistics(
                image,
                band_names,
                window_size
            )
        )

        # Sample statistics at field locations.
        samples = (
            stats_image
            .sampleRegions(
                collection=sampling_points,
                scale=30,
                geometries=False
            )
        )

        # -------------------------------------------
        # CALCULATE COEFFICIENTS OF VARIATION
        # -------------------------------------------

        def add_metadata(feature):

            updated = feature

            for band in band_names:

                mean_value = feature.get(
                    f"{band}_mean"
                )

                median_value = feature.get(
                    f"{band}_median"
                )

                std_value = feature.get(
                    f"{band}_stdDev"
                )

                # CV relative to neighborhood mean
                cv_mean = ee.Algorithms.If(
                    ee.Algorithms.IsEqual(
                        mean_value,
                        None
                    ),
                    None,
                    ee.Algorithms.If(
                        ee.Number(
                            mean_value
                        ).neq(0),
                        ee.Number(
                            std_value
                        )
                        .divide(
                            mean_value
                        )
                        .multiply(100),
                        None
                    )
                )

                # CV relative to neighborhood median
                cv_median = ee.Algorithms.If(
                    ee.Algorithms.IsEqual(
                        median_value,
                        None
                    ),
                    None,
                    ee.Algorithms.If(
                        ee.Number(
                            median_value
                        ).neq(0),
                        ee.Number(
                            std_value
                        )
                        .divide(
                            median_value
                        )
                        .multiply(100),
                        None
                    )
                )

                updated = updated.set({
                    f"{band}_cv_mean":
                        cv_mean,

                    f"{band}_cv_median":
                        cv_median
                })

            return updated.set({
                "image_id":
                    image_id,

                "date":
                    acquisition_date,

                "year":
                    year,

                "window_size":
                    window_size
            })

        return samples.map(
            add_metadata
        )

    return (
        collection_year
        .map(process_image)
        .flatten()
    )


# ===================================================
# RUN EXTRACTION
# ===================================================

results = {}

PRODUCT_BANDS = {
    "TOA": TOA_BANDS,
    "SR": SR_BANDS
}


for product in DATA_PRODUCTS:

    if product not in landsat_collections:
        continue

    collection = (
        landsat_collections[
            product
        ]
    )

    band_names = (
        PRODUCT_BANDS[
            product
        ]
    )

    for window_size in WINDOW_SIZES:

        dataset_name = (
            f"{product}_"
            f"{window_size}x{window_size}"
        )

        print(
            f"\nProcessing "
            f"{dataset_name}"
        )

        yearly_data = []

        for year in YEARS:

            print(
                f"  Year {year}"
            )

            fc_year = (
                extract_spectral_statistics(
                    collection=
                        collection,

                    band_names=
                        band_names,

                    window_size=
                        window_size,

                    year=
                        year,

                    sampling_points=
                        points_fc
                )
            )

            df_year = (
                fc_to_df(
                    fc_year
                )
            )

            if not df_year.empty:
                yearly_data.append(
                    df_year
                )

        if yearly_data:

            results[
                dataset_name
            ] = pd.concat(
                yearly_data,
                ignore_index=True
            )

            print(
                f"  Extracted records: "
                f"{len(results[dataset_name])}"
            )

        else:

            print(
                "  No valid observations "
                "were extracted."
            )

In [ ]:
#@title 2.7. Export Spectral Datasets

# ===================================================
# OUTPUT FILE
# ===================================================

SPECTRAL_OUTPUT_FILE = os.path.join(
    OUTPUT_DIR,
    (
        f"Landsat8-9_Spectral_Extraction_"
        f"{START_YEAR}-{END_YEAR}.xlsx"
    )
)


# ===================================================
# EXPORT EACH PRODUCT / WINDOW AS A SHEET
# ===================================================

with pd.ExcelWriter(
    SPECTRAL_OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    for dataset_name, df in results.items():

        print(
            f"Preparing {dataset_name}..."
        )

        # Identify neighborhood mean columns.
        mean_columns = [
            column
            for column in df.columns
            if column.endswith(
                "_mean"
            )
        ]

        # Remove records where no spectral
        # information was available.
        if mean_columns:

            df_clean = (
                df.dropna(
                    subset=mean_columns,
                    how="all"
                )
                .copy()
            )

        else:

            df_clean = df.copy()

        print(
            f"  Before cleaning: "
            f"{len(df)}"
        )

        print(
            f"  After cleaning : "
            f"{len(df_clean)}"
        )

        # Excel sheet names must be <=31 characters.
        df_clean.to_excel(
            writer,
            sheet_name=
                dataset_name[:31],
            index=False
        )


print(
    "\nSpectral extraction complete."
)

print(
    f"Output saved to:\n"
    f"{SPECTRAL_OUTPUT_FILE}"
)

In [ ]:
#@title 🛰️📍 3. LANDSAT AND IN-SITU MATCH-UP

This section temporally matches the Landsat spectral observations with the processed in-situ water-quality measurements. Matchups are performed independently for each sampling location using a user-defined temporal window around the satellite acquisition date.

The default configuration retains field observations collected within ±8 days of a Landsat overpass. Users may modify this temporal tolerance according to the characteristics of their monitoring dataset.

An optional coefficient-of-variation (CV) filter can be applied to remove spatially heterogeneous satellite observations. When enabled, a matchup is retained only when the mean-based CV of all Landsat spectral bands is less than or equal to the specified threshold. The default threshold used in this workflow is 20%.

Separate matchup datasets are generated for SDD, Chl-a, and paired SDD–Chl-a observations for each Landsat product and spatial window.

In [ ]:
#@title 3.1. Configure Matchup

# ===================================================
# TEMPORAL MATCHUP WINDOW
# ===================================================

# Maximum number of days allowed between the
# satellite acquisition and field observation.
MATCHUP_DAYS = 8

# ===================================================
# OPTIONAL CV FILTERING
# ===================================================

# Apply a coefficient-of-variation filter to the
# extracted Landsat observations.
APPLY_CV_FILTER = True

# Maximum allowable CV (%) for each spectral band.
CV_THRESHOLD = 20.0

# ===================================================
# TARGET VARIABLES
# ===================================================

TARGET_VARIABLES = [
    "sdd",
    "chl_a"
]

print("Satellite–field matchup configuration")
print("--------------------------------------")
print(f"Temporal window : ±{MATCHUP_DAYS} days")

if APPLY_CV_FILTER:
    print(
        f"CV filtering   : Enabled "
        f"(≤ {CV_THRESHOLD}%)"
    )
else:
    print("CV filtering   : Disabled")

In [ ]:
#@title 3.2. Load and Standardize In-situ Observations

# ===================================================
# READ ALL LAKE SHEETS
# ===================================================

insitu_sheets = pd.read_excel(
    IN_SITU_FILE,
    sheet_name=None
)

insitu_data = []

for lake, df in insitu_sheets.items():

    df = df.copy()

    # Standardize column names
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
    )

    # Retain lake identifier
    df["lake"] = lake

    insitu_data.append(df)


insitu_df = pd.concat(
    insitu_data,
    ignore_index=True
)


# ===================================================
# STANDARDIZE REQUIRED FIELDS
# ===================================================

insitu_df["event_datetime"] = pd.to_datetime(
    insitu_df["event_datetime"],
    errors="coerce"
)

insitu_df["site_code"] = (
    insitu_df["site_code"]
    .astype(str)
    .str.strip()
)

# Remove observations without valid site or date
insitu_df = insitu_df.dropna(
    subset=[
        "event_datetime",
        "site_code"
    ]
)

print(
    f"In-situ observations loaded: "
    f"{len(insitu_df)}"
)

print(
    f"Sampling locations: "
    f"{insitu_df['site_code'].nunique()}"
)

In [ ]:
#@title 3.3. Define Satellite–Field Matchup Function

def perform_matchup(
    satellite_df,
    insitu_df,
    product,
    matchup_days=MATCHUP_DAYS,
    apply_cv_filter=APPLY_CV_FILTER,
    cv_threshold=CV_THRESHOLD
):
    """
    Match Landsat spectral observations with in-situ
    water-quality measurements at the same sampling
    location within a specified temporal window.

    Parameters
    ----------
    satellite_df : pandas.DataFrame
        Extracted Landsat spectral observations.

    insitu_df : pandas.DataFrame
        Processed field observations.

    product : str
        Landsat product ("TOA" or "SR").

    matchup_days : int
        Maximum absolute number of days between
        satellite and field observations.

    apply_cv_filter : bool
        Whether to apply the spectral CV filter.

    cv_threshold : float
        Maximum allowable CV (%) for every band.

    Returns
    -------
    pandas.DataFrame
        Satellite–field matchup dataset.
    """

    sat_df = satellite_df.copy()

    # ===================================================
    # STANDARDIZE SATELLITE FIELDS
    # ===================================================

    sat_df["date"] = pd.to_datetime(
        sat_df["date"],
        errors="coerce"
    )

    sat_df["site_code"] = (
        sat_df["site_code"]
        .astype(str)
        .str.strip()
    )

    sat_df = sat_df.dropna(
        subset=[
            "date",
            "site_code"
        ]
    )

    # ===================================================
    # OPTIONAL CV FILTERING
    # ===================================================

    if apply_cv_filter:

        if product == "TOA":

            cv_columns = [
                f"B{i}_cv_mean"
                for i in range(1, 8)
            ]

        elif product == "SR":

            cv_columns = [
                f"SR_B{i}_cv_mean"
                for i in range(1, 8)
            ]

        else:
            raise ValueError(
                "Product must be 'TOA' or 'SR'."
            )

        # Check that all required CV columns exist.
        missing_cv = [
            column
            for column in cv_columns
            if column not in sat_df.columns
        ]

        if missing_cv:
            raise ValueError(
                "Missing CV columns: "
                + ", ".join(missing_cv)
            )

        before_filter = len(sat_df)

        # Retain records where every spectral band
        # satisfies the selected CV threshold.
        sat_df = sat_df[
            (
                sat_df[cv_columns]
                <= cv_threshold
            ).all(axis=1)
        ].copy()

        print(
            f"CV filtering: "
            f"{before_filter} → {len(sat_df)} records"
        )

    # ===================================================
    # PREPARE DATA FOR MATCHING
    # ===================================================

    sat_df = sat_df.sort_values(
        ["site_code", "date"]
    )

    field_df = insitu_df.sort_values(
        [
            "site_code",
            "event_datetime"
        ]
    )

    matchup_rows = []

    # ===================================================
    # TEMPORAL MATCHUP
    # ===================================================

    for _, sat_row in sat_df.iterrows():

        site = sat_row["site_code"]
        satellite_date = sat_row["date"]

        site_field = field_df[
            field_df["site_code"] == site
        ]

        if site_field.empty:
            continue

        lower_date = (
            satellite_date
            - pd.Timedelta(
                days=matchup_days
            )
        )

        upper_date = (
            satellite_date
            + pd.Timedelta(
                days=matchup_days
            )
        )

        matched_field = site_field[
            (
                site_field[
                    "event_datetime"
                ] >= lower_date
            )
            &
            (
                site_field[
                    "event_datetime"
                ] <= upper_date
            )
        ]

        if matched_field.empty:
            continue

        # ===================================================
        # CREATE MATCHUP RECORDS
        # ===================================================

        for _, field_row in matched_field.iterrows():

            matchup_record = (
                sat_row.to_dict()
            )

            matchup_record[
                "matched_date"
            ] = field_row[
                "event_datetime"
            ]

            matchup_record[
                "days_difference"
            ] = abs(
                (
                    satellite_date.date()
                    -
                    field_row[
                        "event_datetime"
                    ].date()
                ).days
            )

            matchup_record[
                "lake"
            ] = field_row.get(
                "lake",
                None
            )

            for variable in TARGET_VARIABLES:

                matchup_record[
                    variable
                ] = field_row.get(
                    variable,
                    np.nan
                )

            matchup_rows.append(
                matchup_record
            )

    matchup_df = pd.DataFrame(
        matchup_rows
    )

    return matchup_df

In [ ]:
#@title  3.4. Run Matchups for All Landsat Datasets

# ===================================================
# OUTPUT DIRECTORY
# ===================================================

MATCHUP_OUTPUT_DIR = os.path.join(
    OUTPUT_DIR,
    "matchups"
)

os.makedirs(
    MATCHUP_OUTPUT_DIR,
    exist_ok=True
)


# ===================================================
# RUN MATCHUPS
# ===================================================

matchup_results = {}


for dataset_name, satellite_df in results.items():

    # Determine Landsat product
    if dataset_name.startswith("TOA"):
        product = "TOA"

    elif dataset_name.startswith("SR"):
        product = "SR"

    else:
        print(
            f"Skipping unrecognized dataset: "
            f"{dataset_name}"
        )
        continue

    print(
        f"\nProcessing matchup: "
        f"{dataset_name}"
    )

    matchup_df = perform_matchup(
        satellite_df=satellite_df,
        insitu_df=insitu_df,
        product=product
    )

    if matchup_df.empty:

        print(
            "No satellite–field "
            "matchups found."
        )

        continue

    # ===================================================
    # TARGET-SPECIFIC DATASETS
    # ===================================================

    sdd_df = matchup_df[
        matchup_df["sdd"].notna()
    ].copy()

    chla_df = matchup_df[
        matchup_df["chl_a"].notna()
    ].copy()

    paired_df = matchup_df[
        matchup_df["sdd"].notna()
        &
        matchup_df["chl_a"].notna()
    ].copy()


    matchup_results[
        dataset_name
    ] = {
        "Master": matchup_df,
        "SDD": sdd_df,
        "CHL_A": chla_df,
        "SDD_CHL_A": paired_df
    }


    # ===================================================
    # EXPORT
    # ===================================================

    cv_suffix = (
        f"_CV{CV_THRESHOLD:g}"
        if APPLY_CV_FILTER
        else ""
    )

    output_file = os.path.join(
        MATCHUP_OUTPUT_DIR,
        (
            f"{dataset_name}_"
            f"Matched{cv_suffix}.xlsx"
        )
    )

    with pd.ExcelWriter(
        output_file,
        engine="openpyxl"
    ) as writer:

        matchup_df.to_excel(
            writer,
            sheet_name="Master",
            index=False
        )

        sdd_df.to_excel(
            writer,
            sheet_name="SDD",
            index=False
        )

        chla_df.to_excel(
            writer,
            sheet_name="CHL_A",
            index=False
        )

        paired_df.to_excel(
            writer,
            sheet_name="SDD_CHL_A",
            index=False
        )


    print(
        f"Master records : {len(matchup_df)}"
    )

    print(
        f"SDD matchups   : {len(sdd_df)}"
    )

    print(
        f"Chl-a matchups : {len(chla_df)}"
    )

    print(
        f"Paired records : {len(paired_df)}"
    )

    print(
        f"Saved to:\n{output_file}"
    )

In [ ]:
#@title 3.5. Summarize Monthly Image and Matchup Availability

# ===================================================
# DATASET USED FOR AVAILABILITY SUMMARY
# ===================================================

SUMMARY_DATASET = (
    f"{DATA_PRODUCT}_"
    f"{WINDOW_SIZES[0]}x{WINDOW_SIZES[0]}"
)

if SUMMARY_DATASET not in results:
    raise ValueError(
        f"{SUMMARY_DATASET} was not generated "
        "during spectral extraction."
    )

if SUMMARY_DATASET not in matchup_results:
    raise ValueError(
        f"No matchup dataset available for "
        f"{SUMMARY_DATASET}."
    )


sat_df = results[
    SUMMARY_DATASET
].copy()

sdd_df = matchup_results[
    SUMMARY_DATASET
]["SDD"].copy()

chla_df = matchup_results[
    SUMMARY_DATASET
]["CHL_A"].copy()


# ===================================================
# STANDARDIZE DATES
# ===================================================

sat_df["date"] = pd.to_datetime(
    sat_df["date"]
)

sdd_df["date"] = pd.to_datetime(
    sdd_df["date"]
)

chla_df["date"] = pd.to_datetime(
    chla_df["date"]
)


# ===================================================
# SATELLITE IMAGE COUNTS
# ===================================================

# Each Landsat image can occur at several sampling
# locations, so count unique acquisition dates
# within each lake.

sat_unique = (
    sat_df[
        [
            "lake",
            "date",
            "image_id"
        ]
    ]
    .drop_duplicates()
    .copy()
)

sat_unique["year_month"] = (
    sat_unique["date"]
    .dt.to_period("M")
)

sat_summary = (
    sat_unique
    .groupby(
        [
            "lake",
            "year_month"
        ]
    )
    .size()
    .reset_index(
        name="images"
    )
)


# ===================================================
# MATCHUP COUNTS
# ===================================================

sdd_df["year_month"] = (
    sdd_df["date"]
    .dt.to_period("M")
)

chla_df["year_month"] = (
    chla_df["date"]
    .dt.to_period("M")
)


sdd_summary = (
    sdd_df
    .groupby(
        [
            "lake",
            "year_month"
        ]
    )
    .size()
    .reset_index(
        name="sdd_matchups"
    )
)


chla_summary = (
    chla_df
    .groupby(
        [
            "lake",
            "year_month"
        ]
    )
    .size()
    .reset_index(
        name="chla_matchups"
    )
)


# ===================================================
# CREATE COMPLETE MONTHLY GRID
# ===================================================

all_months = pd.period_range(
    start=pd.Timestamp(
        START_YEAR,
        1,
        1
    ),
    end=pd.Timestamp(
        END_YEAR,
        12,
        31
    ),
    freq="M"
)

# Restrict monthly summary to the same seasonal
# window used for Landsat extraction.
start_month = pd.Timestamp(
    2000,
    1,
    1
) + pd.Timedelta(
    days=START_DOY - 1
)

end_month = pd.Timestamp(
    2000,
    1,
    1
) + pd.Timedelta(
    days=END_DOY - 1
)

valid_months = list(
    range(
        start_month.month,
        end_month.month + 1
    )
)

all_months = all_months[
    all_months.month.isin(
        valid_months
    )
]


lakes = sorted(
    sat_unique["lake"]
    .dropna()
    .unique()
)


full_grid = (
    pd.MultiIndex
    .from_product(
        [
            lakes,
            all_months
        ],
        names=[
            "lake",
            "year_month"
        ]
    )
    .to_frame(
        index=False
    )
)


# ===================================================
# MERGE SUMMARY TABLES
# ===================================================

monthly_summary = (
    full_grid
    .merge(
        sat_summary,
        on=[
            "lake",
            "year_month"
        ],
        how="left"
    )
    .merge(
        sdd_summary,
        on=[
            "lake",
            "year_month"
        ],
        how="left"
    )
    .merge(
        chla_summary,
        on=[
            "lake",
            "year_month"
        ],
        how="left"
    )
)


count_columns = [
    "images",
    "sdd_matchups",
    "chla_matchups"
]

monthly_summary[
    count_columns
] = (
    monthly_summary[
        count_columns
    ]
    .fillna(0)
    .astype(int)
)


monthly_summary["date"] = (
    monthly_summary[
        "year_month"
    ]
    .dt.to_timestamp()
)


monthly_summary = (
    monthly_summary[
        [
            "lake",
            "date",
            "images",
            "sdd_matchups",
            "chla_matchups"
        ]
    ]
    .sort_values(
        [
            "lake",
            "date"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ===================================================
# EXPORT
# ===================================================

MONTHLY_MATCHUP_SUMMARY_FILE = os.path.join(
    MATCHUP_OUTPUT_DIR,
    (
        f"{SUMMARY_DATASET}_"
        f"Monthly_Matchup_Summary.xlsx"
    )
)

monthly_summary.to_excel(
    MONTHLY_MATCHUP_SUMMARY_FILE,
    index=False
)

display(
    monthly_summary.head()
)

print(
    f"Monthly matchup summary saved to:\n"
    f"{MONTHLY_MATCHUP_SUMMARY_FILE}"
)

In [ ]:
#@title 🧩 4. FEATURE GENERATION AND SELECTION

This section generates a comprehensive set of candidate spectral predictors from the Landsat bands and implements a training-only feature-selection workflow.

Candidate features include the original spectral bands, logarithmic transformations, pairwise band ratios, normalized differences, pairwise additions and subtractions, and three-band combinations. Feature generation is performed independently for each observation and therefore does not use information from the target variable.

Feature selection is performed only using training observations to prevent information leakage into model evaluation. The selection procedure removes features with excessive missing values, imputes remaining missing values using training-set medians, removes near-constant and redundant features, ranks predictors using their Spearman correlation with the target variable, and applies cross-validated LASSO regression for final feature selection.

The same training-only selection procedure is subsequently repeated within each Monte Carlo model evaluation.

In [ ]:
#@title 4.1. Configure Feature Engineering

# ===================================================
# DATASET
# ===================================================

# Landsat product:
# "TOA" or "SR"
DATA_SOURCE = "TOA"

# Spatial window:
# "3x3", "5x5", or "7x7"
WINDOW_SIZE = "3x3"

# Water-quality target:
# "sdd" or "chl_a"
TARGET = "sdd"

# ===================================================
# TRAIN / TEST SPLIT
# ===================================================

TEST_SIZE = 0.20
RANDOM_STATE = 42

# Stratify the continuous target using quantile bins.
USE_QUANTILE_SPLIT = True
N_QUANTILES = 5

# ===================================================
# FEATURE-SELECTION SETTINGS
# ===================================================

# Maximum fraction of missing values allowed for a feature.
MAX_NAN_FRACTION = 0.10

# Variance threshold for removing near-constant features.
VARIANCE_THRESHOLD = 1e-6

# Number of features retained after initial
# target-correlation ranking.
TOP_N_CORRELATED = 1500

# Maximum allowable absolute Spearman correlation
# between candidate predictors.
FEATURE_CORRELATION_THRESHOLD = 0.95

# Maximum number of predictors supplied to LASSO.
TOP_N_LASSO = 100

# LASSO regularization strengths.
LASSO_ALPHAS = [
    0.001,
    0.005,
    0.01,
    0.02,
    0.05,
    0.1
]

LASSO_CV_FOLDS = 5

# ===================================================
# FEATURE OUTPUT DIRECTORY
# ===================================================

FEATURE_OUTPUT_DIR = os.path.join(
    OUTPUT_DIR,
    "features"
)

os.makedirs(
    FEATURE_OUTPUT_DIR,
    exist_ok=True
)

# ===================================================
# DATASET IDENTIFIER
# ===================================================

DATASET_NAME = (
    f"{DATA_SOURCE}_{WINDOW_SIZE}"
)

if TARGET == "sdd":
    TARGET_SHEET = "SDD"

elif TARGET == "chl_a":
    TARGET_SHEET = "CHL_A"

else:
    raise ValueError(
        "TARGET must be 'sdd' or 'chl_a'."
    )

print("Feature-engineering configuration")
print("---------------------------------")
print(f"Dataset              : {DATASET_NAME}")
print(f"Target               : {TARGET}")
print(f"Test fraction        : {TEST_SIZE}")
print(f"Quantile stratify    : {USE_QUANTILE_SPLIT}")
print(f"Quantile groups      : {N_QUANTILES}")
print(f"Maximum NaN fraction : {MAX_NAN_FRACTION}")
print(f"Variance threshold   : {VARIANCE_THRESHOLD}")
print(f"Initial top features : {TOP_N_CORRELATED}")
print(
    f"Correlation threshold: "
    f"{FEATURE_CORRELATION_THRESHOLD}"
)
print(f"Maximum LASSO input  : {TOP_N_LASSO}")

In [ ]:
#@title 4.2. Prepare Matchup Dataset

# ===================================================
# LOAD MATCHUP DATA
# ===================================================

if DATASET_NAME not in matchup_results:
    raise ValueError(
        f"{DATASET_NAME} is not available in "
        "the matchup results."
    )

data = (
    matchup_results[
        DATASET_NAME
    ][TARGET_SHEET]
    .copy()
)

data.columns = (
    data.columns
    .str.strip()
)

# ===================================================
# STANDARDIZE SPECTRAL BAND NAMES
# ===================================================

if DATA_SOURCE == "TOA":

    band_rename = {
        f"B{i}_mean": f"B{i}"
        for i in range(1, 8)
    }

elif DATA_SOURCE == "SR":

    band_rename = {
        f"SR_B{i}_mean": f"B{i}"
        for i in range(1, 8)
    }

else:
    raise ValueError(
        "DATA_SOURCE must be 'TOA' or 'SR'."
    )

data = data.rename(
    columns=band_rename
)

BANDS = [
    f"B{i}"
    for i in range(1, 8)
]

# ===================================================
# CHECK REQUIRED FIELDS
# ===================================================

required_columns = (
    BANDS
    + [TARGET]
)

missing_columns = [
    column
    for column in required_columns
    if column not in data.columns
]

if missing_columns:
    raise ValueError(
        "Required columns are missing: "
        + ", ".join(missing_columns)
    )

# ===================================================
# CONVERT SPECTRAL VALUES TO NUMERIC
# ===================================================

for band in BANDS:

    data[band] = pd.to_numeric(
        data[band],
        errors="coerce"
    )

data[TARGET] = pd.to_numeric(
    data[TARGET],
    errors="coerce"
)

# Preserve original matchup row.
data = data.reset_index(
    drop=True
)

data["original_row_index"] = (
    data.index
)

# Remove only observations without the target.
data = (
    data
    .dropna(
        subset=[TARGET]
    )
    .reset_index(
        drop=True
    )
)

y = data[TARGET].copy()

print(
    "Observations available:",
    len(data)
)

print(
    "Target range:",
    f"{y.min():.3f} – {y.max():.3f}"
)

print(
    "Missing spectral values:",
    data[BANDS].isna().sum().sum()
)

In [ ]:
#@title 4.3. Generate Candidate Spectral Features

def generate_spectral_features(
    input_data,
    bands
):
    """
    Generate candidate spectral predictors from
    Landsat bands.

    Feature generation is row-wise and does not
    use the target variable or statistics calculated
    across observations.
    """

    feature_dict = {}

    epsilon = 1e-10

    # ===================================================
    # 1. RAW AND LOG-TRANSFORMED BANDS
    # ===================================================

    base_inputs = {}

    for band_name in bands:

        band = input_data[
            band_name
        ].astype(float)

        feature_dict[
            band_name
        ] = band

        base_inputs[
            band_name
        ] = band

        log_name = (
            f"log_{band_name}"
        )

        log_band = np.log(
            band.clip(
                lower=1e-6
            )
        )

        feature_dict[
            log_name
        ] = log_band

        base_inputs[
            log_name
        ] = log_band

    # ===================================================
    # 2. BAND RATIOS AND LOG RATIOS
    # ===================================================

    for b1, b2 in permutations(
        bands,
        2
    ):

        ratio_name = (
            f"{b1}_div_{b2}"
        )

        ratio = (
            input_data[b1]
            /
            (
                input_data[b2]
                + epsilon
            )
        )

        feature_dict[
            ratio_name
        ] = ratio

        base_inputs[
            ratio_name
        ] = ratio

        log_ratio_name = (
            f"log_{b1}_div_{b2}"
        )

        log_ratio = np.log(
            ratio.clip(
                lower=1e-6
            )
        )

        feature_dict[
            log_ratio_name
        ] = log_ratio

        base_inputs[
            log_ratio_name
        ] = log_ratio

    # ===================================================
    # 3. PAIRWISE ADDITION AND SUBTRACTION
    # ===================================================

    base_names = list(
        base_inputs.keys()
    )

    for f1, f2 in combinations(
        base_names,
        2
    ):

        x = base_inputs[f1]
        z = base_inputs[f2]

        feature_dict[
            f"{f1}_plus_{f2}"
        ] = x + z

        feature_dict[
            f"{f1}_minus_{f2}"
        ] = x - z

        feature_dict[
            f"{f2}_minus_{f1}"
        ] = z - x

    # ===================================================
    # 4. NORMALIZED DIFFERENCES
    # ===================================================

    for b1, b2 in combinations(
        bands,
        2
    ):

        feature_dict[
            f"{b1}_ND_{b2}"
        ] = (
            (
                input_data[b1]
                - input_data[b2]
            )
            /
            (
                input_data[b1]
                + input_data[b2]
                + epsilon
            )
        )

    # ===================================================
    # 5. THREE-BAND COMBINATIONS
    # ===================================================

    for Bi, Bj, Bk in permutations(
        bands,
        3
    ):

        ratio = (
            input_data[Bi]
            /
            (
                input_data[Bj]
                + epsilon
            )
        )

        log_ratio = np.log(
            ratio.clip(
                lower=1e-6
            )
        )

        band = input_data[Bk]

        log_band = np.log(
            band.clip(
                lower=1e-6
            )
        )

        nd = (
            (
                input_data[Bi]
                - input_data[Bj]
            )
            /
            (
                input_data[Bi]
                + input_data[Bj]
                + epsilon
            )
        )

        # -----------------------------------------------
        # ratio ± band
        # -----------------------------------------------

        feature_dict[
            f"({Bi}_div_{Bj})_plus_{Bk}"
        ] = ratio + band

        feature_dict[
            f"({Bi}_div_{Bj})_minus_{Bk}"
        ] = ratio - band

        # -----------------------------------------------
        # log ratio ± band
        # -----------------------------------------------

        feature_dict[
            f"(log_{Bi}_div_{Bj})_plus_{Bk}"
        ] = log_ratio + band

        feature_dict[
            f"(log_{Bi}_div_{Bj})_minus_{Bk}"
        ] = log_ratio - band

        # -----------------------------------------------
        # ratio ± log band
        # -----------------------------------------------

        feature_dict[
            f"({Bi}_div_{Bj})_plus_log_{Bk}"
        ] = ratio + log_band

        feature_dict[
            f"({Bi}_div_{Bj})_minus_log_{Bk}"
        ] = ratio - log_band

        # -----------------------------------------------
        # log ratio ± log band
        # -----------------------------------------------

        feature_dict[
            f"(log_{Bi}_div_{Bj})_plus_log_{Bk}"
        ] = (
            log_ratio
            + log_band
        )

        feature_dict[
            f"(log_{Bi}_div_{Bj})_minus_log_{Bk}"
        ] = (
            log_ratio
            - log_band
        )

        # -----------------------------------------------
        # normalized difference ± band
        # -----------------------------------------------

        feature_dict[
            f"({Bi}_ND_{Bj})_plus_{Bk}"
        ] = nd + band

        feature_dict[
            f"({Bi}_ND_{Bj})_minus_{Bk}"
        ] = nd - band

        # -----------------------------------------------
        # normalized difference ± log band
        # -----------------------------------------------

        feature_dict[
            f"({Bi}_ND_{Bj})_plus_log_{Bk}"
        ] = (
            nd
            + log_band
        )

        feature_dict[
            f"({Bi}_ND_{Bj})_minus_log_{Bk}"
        ] = (
            nd
            - log_band
        )

    # ===================================================
    # 6. RATIO–RATIO COMBINATIONS
    # ===================================================

    for Bi, Bj, Bk in permutations(
        bands,
        3
    ):

        ratio_ij = (
            input_data[Bi]
            /
            (
                input_data[Bj]
                + epsilon
            )
        )

        ratio_ik = (
            input_data[Bi]
            /
            (
                input_data[Bk]
                + epsilon
            )
        )

        log_ratio_ij = np.log(
            ratio_ij.clip(
                lower=1e-6
            )
        )

        log_ratio_ik = np.log(
            ratio_ik.clip(
                lower=1e-6
            )
        )

        # -----------------------------------------------
        # ratio ± ratio
        # -----------------------------------------------

        feature_dict[
            f"({Bi}_div_{Bj})_plus_"
            f"({Bi}_div_{Bk})"
        ] = (
            ratio_ij
            + ratio_ik
        )

        feature_dict[
            f"({Bi}_div_{Bj})_minus_"
            f"({Bi}_div_{Bk})"
        ] = (
            ratio_ij
            - ratio_ik
        )

        # -----------------------------------------------
        # log ratio ± log ratio
        # -----------------------------------------------

        feature_dict[
            f"(log_{Bi}_div_{Bj})_plus_"
            f"(log_{Bi}_div_{Bk})"
        ] = (
            log_ratio_ij
            + log_ratio_ik
        )

        feature_dict[
            f"(log_{Bi}_div_{Bj})_minus_"
            f"(log_{Bi}_div_{Bk})"
        ] = (
            log_ratio_ij
            - log_ratio_ik
        )

        # -----------------------------------------------
        # ratio ± log ratio
        # -----------------------------------------------

        feature_dict[
            f"({Bi}_div_{Bj})_plus_"
            f"(log_{Bi}_div_{Bk})"
        ] = (
            ratio_ij
            + log_ratio_ik
        )

        feature_dict[
            f"({Bi}_div_{Bj})_minus_"
            f"(log_{Bi}_div_{Bk})"
        ] = (
            ratio_ij
            - log_ratio_ik
        )

    # ===================================================
    # CONVERT TO DATAFRAME
    # ===================================================

    feature_df = pd.DataFrame(
        feature_dict,
        index=input_data.index
    )

    feature_df = feature_df.replace(
        [np.inf, -np.inf],
        np.nan
    )

    return feature_df


feature_df = generate_spectral_features(
    input_data=data,
    bands=BANDS
)

print(
    "Candidate feature matrix:",
    feature_df.shape
)

In [ ]:
#@title 4.4. Create Reference Training and Test Split

def create_train_test_indices(
    target,
    test_size=0.20,
    random_state=42,
    use_quantiles=True,
    n_quantiles=5
):
    """
    Create training and test indices.

    When requested, continuous target values are
    divided into quantile groups and used for
    stratification.
    """

    indices = np.arange(
        len(target)
    )

    stratify_values = None

    if use_quantiles:

        try:

            quantile_groups = pd.qcut(
                target,
                q=n_quantiles,
                labels=False,
                duplicates="drop"
            )

            group_counts = (
                pd.Series(
                    quantile_groups
                )
                .value_counts()
            )

            # Stratification requires at least two
            # observations in every group.
            if (
                len(group_counts) >= 2
                and group_counts.min() >= 2
            ):

                stratify_values = (
                    quantile_groups
                )

            else:

                print(
                    "Quantile stratification could "
                    "not be applied safely."
                )

        except ValueError:

            print(
                "Quantile stratification failed. "
                "Using random split instead."
            )

    train_idx, test_idx = train_test_split(
        indices,
        test_size=test_size,
        random_state=random_state,
        stratify=stratify_values
    )

    return train_idx, test_idx


train_idx, test_idx = (
    create_train_test_indices(
        target=y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        use_quantiles=USE_QUANTILE_SPLIT,
        n_quantiles=N_QUANTILES
    )
)


X_train_full = (
    feature_df
    .iloc[train_idx]
    .copy()
)

X_test_full = (
    feature_df
    .iloc[test_idx]
    .copy()
)

y_train = (
    y.iloc[train_idx]
    .copy()
)

y_test = (
    y.iloc[test_idx]
    .copy()
)


print(
    f"Training observations: "
    f"{len(train_idx)}"
)

print(
    f"Test observations    : "
    f"{len(test_idx)}"
)

In [ ]:
#@title 4.5. Define Training-Only Feature Selection

def select_features_from_training(
    X_train,
    y_train,
    max_nan_fraction=MAX_NAN_FRACTION,
    variance_threshold=VARIANCE_THRESHOLD,
    top_n_correlated=TOP_N_CORRELATED,
    correlation_threshold=FEATURE_CORRELATION_THRESHOLD,
    top_n_lasso=TOP_N_LASSO,
    lasso_alphas=LASSO_ALPHAS,
    lasso_cv=LASSO_CV_FOLDS
):
    """
    Fit the complete feature-selection pipeline
    using training observations only.

    Returns
    -------
    X_train_selected : DataFrame
        Selected training predictors.

    selector : dict
        Fitted preprocessing and selection objects
        required to transform unseen data.

    selected_summary : DataFrame
        Summary of final selected predictors.
    """

    X = X_train.copy()

    X = X.replace(
        [np.inf, -np.inf],
        np.nan
    )

    selection_log = []

    # ===================================================
    # STEP 1. REMOVE FEATURES WITH EXCESSIVE NaN
    # ===================================================

    initial_count = X.shape[1]

    nan_fraction = (
        X.isna().mean()
    )

    nan_eligible_columns = (
        nan_fraction[
            nan_fraction
            <= max_nan_fraction
        ]
        .index
        .tolist()
    )

    X = X[
        nan_eligible_columns
    ].copy()

    selection_log.append({
        "Step": "Initial features",
        "Feature_Count": initial_count
    })

    selection_log.append({
        "Step": (
            f"NaN fraction ≤ "
            f"{max_nan_fraction}"
        ),
        "Feature_Count": X.shape[1]
    })

    # ===================================================
    # STEP 2. TRAINING-MEDIAN IMPUTATION
    # ===================================================

    imputer = SimpleImputer(
        strategy="median"
    )

    X_imputed = pd.DataFrame(
        imputer.fit_transform(X),
        columns=X.columns,
        index=X.index
    )

    # ===================================================
    # STEP 3. REMOVE EXACT DUPLICATE FEATURES
    # ===================================================

    duplicate_mask = (
        X_imputed.T.duplicated()
    )

    nonduplicate_columns = (
        X_imputed.columns[
            ~duplicate_mask
        ]
        .tolist()
    )

    X_imputed = X_imputed[
        nonduplicate_columns
    ].copy()

    selection_log.append({
        "Step": "Remove duplicate features",
        "Feature_Count": X_imputed.shape[1]
    })

    # ===================================================
    # STEP 4. REMOVE NEAR-CONSTANT FEATURES
    # ===================================================

    variance_filter = (
        VarianceThreshold(
            threshold=variance_threshold
        )
    )

    X_variance_array = (
        variance_filter
        .fit_transform(
            X_imputed
        )
    )

    variance_columns = (
        X_imputed.columns[
            variance_filter
            .get_support()
        ]
        .tolist()
    )

    X_variance = pd.DataFrame(
        X_variance_array,
        columns=variance_columns,
        index=X_imputed.index
    )

    selection_log.append({
        "Step": "Variance filter",
        "Feature_Count": X_variance.shape[1]
    })

    # ===================================================
    # STEP 5. TARGET-CORRELATION RANKING
    # ===================================================

    target_corr_initial = (
        X_variance
        .corrwith(
            y_train,
            method="spearman"
        )
        .abs()
        .dropna()
        .sort_values(
            ascending=False
        )
    )

    n_initial = min(
        top_n_correlated,
        len(target_corr_initial)
    )

    initial_ranked_features = (
        target_corr_initial
        .head(n_initial)
        .index
        .tolist()
    )

    X_ranked = X_variance[
        initial_ranked_features
    ].copy()

    selection_log.append({
        "Step": (
            f"Top {n_initial} "
            "target-correlated"
        ),
        "Feature_Count": X_ranked.shape[1]
    })

    # ===================================================
    # STEP 6. REMOVE HIGHLY CORRELATED PREDICTORS
    # ===================================================

    feature_corr_matrix = (
        X_ranked
        .corr(
            method="spearman"
        )
        .abs()
    )

    upper_triangle = (
        feature_corr_matrix.where(
            np.triu(
                np.ones(
                    feature_corr_matrix.shape
                ),
                k=1
            ).astype(bool)
        )
    )

    correlated_to_drop = [
        column
        for column
        in upper_triangle.columns
        if (
            upper_triangle[column]
            > correlation_threshold
        ).any()
    ]

    X_uncorrelated = (
        X_ranked
        .drop(
            columns=correlated_to_drop
        )
        .copy()
    )

    selection_log.append({
        "Step": (
            f"Feature correlation ≤ "
            f"{correlation_threshold}"
        ),
        "Feature_Count":
            X_uncorrelated.shape[1]
    })

    # ===================================================
    # STEP 7. SECOND TARGET RANKING
    # ===================================================

    target_corr_final = (
        X_uncorrelated
        .corrwith(
            y_train,
            method="spearman"
        )
        .abs()
        .dropna()
        .sort_values(
            ascending=False
        )
    )

    n_lasso = min(
        top_n_lasso,
        len(target_corr_final)
    )

    lasso_input_features = (
        target_corr_final
        .head(n_lasso)
        .index
        .tolist()
    )

    X_lasso_input = (
        X_uncorrelated[
            lasso_input_features
        ]
        .copy()
    )

    selection_log.append({
        "Step": (
            f"Top {n_lasso} "
            "before LASSO"
        ),
        "Feature_Count":
            X_lasso_input.shape[1]
    })

    # ===================================================
    # STEP 8. STANDARDIZE FOR LASSO
    # ===================================================

    scaler = StandardScaler()

    X_scaled = (
        scaler.fit_transform(
            X_lasso_input
        )
    )

    # ===================================================
    # STEP 9. CROSS-VALIDATED LASSO
    # ===================================================

    lasso = LassoCV(
        cv=lasso_cv,
        alphas=lasso_alphas,
        max_iter=10000,
        n_jobs=-1,
        random_state=42
    )

    lasso.fit(
        X_scaled,
        y_train
    )

    coefficients = pd.Series(
        lasso.coef_,
        index=lasso_input_features
    )

    selected_features = (
        coefficients[
            coefficients != 0
        ]
        .index
        .tolist()
    )

    if len(selected_features) == 0:
        raise ValueError(
            "LASSO selected no predictors. "
            "Review the feature-selection settings."
        )

    X_train_selected = (
        X_lasso_input[
            selected_features
        ]
        .copy()
    )

    selection_log.append({
        "Step": "LASSO selected",
        "Feature_Count":
            len(selected_features)
    })

    # ===================================================
    # FINAL FEATURE SUMMARY
    # ===================================================

    selected_summary = pd.DataFrame({
        "Feature":
            selected_features,

        "LASSO_Coefficient":
            coefficients[
                selected_features
            ].values,

        "Abs_LASSO_Coefficient":
            coefficients[
                selected_features
            ].abs().values,

        "Abs_Spearman_Target_Corr":
            target_corr_final[
                selected_features
            ].values
    })

    selected_summary = (
        selected_summary
        .sort_values(
            "Abs_LASSO_Coefficient",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )

    # ===================================================
    # STORE FITTED PIPELINE
    # ===================================================

    selector = {
        "nan_eligible_columns":
            nan_eligible_columns,

        "imputer":
            imputer,

        "nonduplicate_columns":
            nonduplicate_columns,

        "variance_filter":
            variance_filter,

        "variance_columns":
            variance_columns,

        "initial_ranked_features":
            initial_ranked_features,

        "correlated_to_drop":
            correlated_to_drop,

        "uncorrelated_features":
            X_uncorrelated.columns.tolist(),

        "target_corr_final":
            target_corr_final,

        "lasso_input_features":
            lasso_input_features,

        "scaler":
            scaler,

        "lasso":
            lasso,

        "selected_features":
            selected_features,

        "selection_log":
            pd.DataFrame(
                selection_log
            )
    }

    return (
        X_train_selected,
        selector,
        selected_summary
    )

In [ ]:
#@title 4.6. Select Training Features and Transform Unseen Test Data

def transform_unseen_features(
    X_new,
    selector
):
    """
    Apply a feature-selection pipeline fitted on
    training observations to unseen observations.

    No fitting or target information is used here.
    """

    X = X_new.copy()

    X = X.replace(
        [np.inf, -np.inf],
        np.nan
    )

    # ===================================================
    # APPLY TRAINING-DERIVED NaN COLUMN FILTER
    # ===================================================

    X = X.reindex(
        columns=selector[
            "nan_eligible_columns"
        ]
    )

    # ===================================================
    # APPLY TRAINING-FITTED MEDIAN IMPUTATION
    # ===================================================

    X_imputed = pd.DataFrame(
        selector[
            "imputer"
        ].transform(X),

        columns=selector[
            "nan_eligible_columns"
        ],

        index=X.index
    )

    # ===================================================
    # APPLY TRAINING-DERIVED DUPLICATE FILTER
    # ===================================================

    X_imputed = X_imputed[
        selector[
            "nonduplicate_columns"
        ]
    ].copy()

    # ===================================================
    # APPLY TRAINING-DERIVED VARIANCE FILTER
    # ===================================================

    X_variance_array = (
        selector[
            "variance_filter"
        ]
        .transform(
            X_imputed
        )
    )

    X_variance = pd.DataFrame(
        X_variance_array,

        columns=selector[
            "variance_columns"
        ],

        index=X.index
    )

    # ===================================================
    # RETAIN FINAL TRAINING-SELECTED FEATURES
    # ===================================================

    X_selected = X_variance[
        selector[
            "selected_features"
        ]
    ].copy()

    return X_selected


# ===================================================
# FIT FEATURE SELECTION USING TRAINING DATA ONLY
# ===================================================

(
    X_train_selected,
    reference_selector,
    reference_selected_summary
) = select_features_from_training(
    X_train=X_train_full,
    y_train=y_train
)


# ===================================================
# TRANSFORM UNSEEN TEST DATA
# ===================================================

X_test_selected = (
    transform_unseen_features(
        X_new=X_test_full,
        selector=reference_selector
    )
)


# ===================================================
# SUMMARY
# ===================================================

display(
    reference_selector[
        "selection_log"
    ]
)

display(
    reference_selected_summary
)

print(
    "Best LASSO alpha:",
    reference_selector[
        "lasso"
    ].alpha_
)

print(
    "\nSelected features:",
    len(
        reference_selector[
            "selected_features"
        ]
    )
)

print(
    "Training matrix:",
    X_train_selected.shape
)

print(
    "Unseen test matrix:",
    X_test_selected.shape
)

In [ ]:
#@title 4.7. Export Reference Feature-Selection Results

OUTPUT_TAG = (
    f"{DATA_SOURCE}_"
    f"{WINDOW_SIZE}_"
    f"{TARGET.upper()}_"
    f"ReferenceSplit"
)

REFERENCE_FEATURE_FILE = os.path.join(
    FEATURE_OUTPUT_DIR,
    f"{OUTPUT_TAG}_Features.xlsx"
)

REFERENCE_SELECTION_FILE = os.path.join(
    FEATURE_OUTPUT_DIR,
    f"{OUTPUT_TAG}_SelectionSummary.xlsx"
)


# ===================================================
# TRAINING / TEST FEATURE DATA
# ===================================================

train_output = (
    X_train_selected
    .reset_index(
        drop=True
    )
)

train_output[TARGET] = (
    y_train
    .reset_index(
        drop=True
    )
)

test_output = (
    X_test_selected
    .reset_index(
        drop=True
    )
)

test_output[TARGET] = (
    y_test
    .reset_index(
        drop=True
    )
)


with pd.ExcelWriter(
    REFERENCE_FEATURE_FILE,
    engine="openpyxl"
) as writer:

    train_output.to_excel(
        writer,
        sheet_name="Training",
        index=False
    )

    test_output.to_excel(
        writer,
        sheet_name="Test",
        index=False
    )


# ===================================================
# FEATURE-SELECTION REPORT
# ===================================================

with pd.ExcelWriter(
    REFERENCE_SELECTION_FILE,
    engine="openpyxl"
) as writer:

    # -----------------------------------------------
    # Run configuration
    # -----------------------------------------------

    settings_df = pd.DataFrame({
        "Setting": [
            "DATA_SOURCE",
            "WINDOW_SIZE",
            "TARGET",
            "TEST_SIZE",
            "RANDOM_STATE",
            "USE_QUANTILE_SPLIT",
            "N_QUANTILES",
            "MAX_NAN_FRACTION",
            "VARIANCE_THRESHOLD",
            "TOP_N_CORRELATED",
            "FEATURE_CORRELATION_THRESHOLD",
            "TOP_N_LASSO",
            "LASSO_CV_FOLDS",
            "BEST_LASSO_ALPHA"
        ],

        "Value": [
            DATA_SOURCE,
            WINDOW_SIZE,
            TARGET,
            TEST_SIZE,
            RANDOM_STATE,
            USE_QUANTILE_SPLIT,
            N_QUANTILES,
            MAX_NAN_FRACTION,
            VARIANCE_THRESHOLD,
            TOP_N_CORRELATED,
            FEATURE_CORRELATION_THRESHOLD,
            TOP_N_LASSO,
            LASSO_CV_FOLDS,
            reference_selector[
                "lasso"
            ].alpha_
        ]
    })

    settings_df.to_excel(
        writer,
        sheet_name="Run_Settings",
        index=False
    )

    # -----------------------------------------------
    # Feature counts after each step
    # -----------------------------------------------

    reference_selector[
        "selection_log"
    ].to_excel(
        writer,
        sheet_name="Selection_Log",
        index=False
    )

    # -----------------------------------------------
    # Final target correlations
    # -----------------------------------------------

    (
        reference_selector[
            "target_corr_final"
        ]
        .rename(
            "Abs_Spearman_Correlation"
        )
        .reset_index()
        .rename(
            columns={
                "index": "Feature"
            }
        )
        .to_excel(
            writer,
            sheet_name="Target_Correlations",
            index=False
        )
    )

    # -----------------------------------------------
    # LASSO input features
    # -----------------------------------------------

    pd.DataFrame({
        "LASSO_Input_Feature":
            reference_selector[
                "lasso_input_features"
            ]
    }).to_excel(
        writer,
        sheet_name="LASSO_Input",
        index=False
    )

    # -----------------------------------------------
    # All LASSO coefficients
    # -----------------------------------------------

    lasso_coefficient_df = pd.DataFrame({
        "Feature":
            reference_selector[
                "lasso_input_features"
            ],

        "LASSO_Coefficient":
            reference_selector[
                "lasso"
            ].coef_
    })

    lasso_coefficient_df[
        "Abs_LASSO_Coefficient"
    ] = (
        lasso_coefficient_df[
            "LASSO_Coefficient"
        ].abs()
    )

    lasso_coefficient_df = (
        lasso_coefficient_df
        .sort_values(
            "Abs_LASSO_Coefficient",
            ascending=False
        )
    )

    lasso_coefficient_df.to_excel(
        writer,
        sheet_name="LASSO_Coefficients",
        index=False
    )

    # -----------------------------------------------
    # Final selected predictors
    # -----------------------------------------------

    reference_selected_summary.to_excel(
        writer,
        sheet_name="Selected_Features",
        index=False
    )


print(
    "Reference feature dataset saved to:\n",
    REFERENCE_FEATURE_FILE
)

print(
    "\nFeature-selection report saved to:\n",
    REFERENCE_SELECTION_FILE
)

In [ ]:
#@title 🌲 5. RANDOM FOREST DEVELOPMENT AND EVALUATION

This section develops and evaluates Random Forest regression models using the candidate spectral features generated in Section 4.

Model evaluation is performed using repeated 80:20 Monte Carlo train–test partitions. For every iteration, feature selection is fitted exclusively to the training observations. Random Forest hyperparameters are subsequently optimized using cross-validation within the same training subset. The withheld test observations therefore remain independent of both feature selection and model optimization.

Model performance is evaluated on the original scale of the water-quality variable using the coefficient of determination (R²), adjusted R², root mean squared error (RMSE), mean absolute error (MAE), bias, regression slope, and range-normalized RMSE (NRMSE_range).

After model evaluation, a final model is developed using the complete dataset for subsequent spatio-temporal prediction and mapping.

In [ ]:
#@title 5.1. Configure Random Forest Evaluation

# ===================================================
# TARGET TRANSFORMATION
# ===================================================

# Model the target on log10 scale.
# Predictions are converted back to the original scale
# before calculating evaluation metrics.
USE_LOG_TARGET = False

# ===================================================
# MONTE CARLO EVALUATION
# ===================================================

N_SIMULATIONS = 100
TEST_SIZE = 0.20

USE_QUANTILE_SPLIT = True
N_QUANTILES = 5

# ===================================================
# RANDOM FOREST HYPERPARAMETER SEARCH
# ===================================================

RF_SEARCH_ITERATIONS = 50
RF_CV_FOLDS = 5

RF_PARAM_DISTRIBUTIONS = {
    "n_estimators": randint(100, 500),
    "max_depth": randint(5, 30),
    "min_samples_split": randint(2, 10),
    "min_samples_leaf": randint(1, 5),
    "max_features": [None, "sqrt", "log2"]
}

# ===================================================
# OUTPUTS
# ===================================================

MODEL_OUTPUT_DIR = os.path.join(
    OUTPUT_DIR,
    "models"
)

os.makedirs(
    MODEL_OUTPUT_DIR,
    exist_ok=True
)

SCENARIO_NAME = (
    f"{DATA_SOURCE}_"
    f"{WINDOW_SIZE}_"
    f"{TARGET.upper()}"
)

if USE_LOG_TARGET:
    SCENARIO_NAME += "_LOG10"

# ===================================================
# LABELS
# ===================================================

if TARGET == "sdd":
    TARGET_LABEL = "SDD"
    TARGET_UNIT = "m"

elif TARGET == "chl_a":
    TARGET_LABEL = "Chl-a"
    TARGET_UNIT = "µg/L"

else:
    raise ValueError(
        "TARGET must be 'sdd' or 'chl_a'."
    )

print("Random Forest configuration")
print("---------------------------")
print(f"Scenario              : {SCENARIO_NAME}")
print(f"Target                : {TARGET_LABEL}")
print(f"Log10 target          : {USE_LOG_TARGET}")
print(f"Monte Carlo runs      : {N_SIMULATIONS}")
print(f"Train/test split      : {1-TEST_SIZE:.0%}/{TEST_SIZE:.0%}")
print(f"Quantile groups       : {N_QUANTILES}")
print(f"RF search iterations  : {RF_SEARCH_ITERATIONS}")
print(f"RF CV folds           : {RF_CV_FOLDS}")

In [ ]:
#@title 5.2. Define Target Transformation and Evaluation Metrics

def transform_target(y_values, use_log=False):
    """
    Transform the model target.

    When use_log=True, only positive observations
    can be retained.
    """

    y_values = pd.Series(
        y_values
    ).astype(float)

    if use_log:

        if (y_values <= 0).any():
            raise ValueError(
                "Log10 target transformation requires "
                "strictly positive target values."
            )

        return np.log10(
            y_values
        )

    return y_values.copy()


def inverse_target(
    values,
    use_log=False
):
    """
    Convert predictions back to the original
    target scale.
    """

    values = np.asarray(
        values,
        dtype=float
    )

    if use_log:
        return 10 ** values

    return values


def calculate_regression_metrics(
    observed,
    predicted,
    n_predictors,
    normalization_range
):
    """
    Calculate regression performance statistics
    on the original target scale.
    """

    observed = np.asarray(
        observed,
        dtype=float
    )

    predicted = np.asarray(
        predicted,
        dtype=float
    )

    n = len(observed)

    r2 = r2_score(
        observed,
        predicted
    )

    # Adjusted R²
    if n > n_predictors + 1:

        adjusted_r2 = (
            1
            - (1 - r2)
            * (n - 1)
            / (
                n
                - n_predictors
                - 1
            )
        )

    else:

        adjusted_r2 = np.nan

    rmse = np.sqrt(
        mean_squared_error(
            observed,
            predicted
        )
    )

    mae = mean_absolute_error(
        observed,
        predicted
    )

    bias = np.mean(
        predicted
        - observed
    )

    if (
        len(np.unique(observed)) > 1
        and len(np.unique(predicted)) > 1
    ):

        slope = np.polyfit(
            observed,
            predicted,
            1
        )[0]

    else:

        slope = np.nan

    # Range-normalized RMSE
    if normalization_range > 0:

        nrmse_range = (
            rmse
            / normalization_range
            * 100
        )

    else:

        nrmse_range = np.nan

    return {
        "R2": r2,
        "Adjusted_R2": adjusted_r2,
        "RMSE": rmse,
        "NRMSE_Range_Percent": nrmse_range,
        "MAE": mae,
        "Bias": bias,
        "Slope": slope,
        "N": n,
        "N_Features": n_predictors
    }

In [ ]:
#@title 5.3. Define Random Forest Hyperparameter Optimization

def optimize_random_forest(
    X_train,
    y_train,
    random_state=42
):
    """
    Optimize Random Forest hyperparameters using
    cross-validation on training observations only.
    """

    rf = RandomForestRegressor(
        random_state=random_state,
        n_jobs=-1
    )

    search = RandomizedSearchCV(
        estimator=rf,
        param_distributions=RF_PARAM_DISTRIBUTIONS,
        n_iter=RF_SEARCH_ITERATIONS,
        cv=RF_CV_FOLDS,
        scoring="r2",
        n_jobs=-1,
        random_state=random_state,
        verbose=0
    )

    search.fit(
        X_train,
        y_train
    )

    return (
        search.best_estimator_,
        search.best_params_,
        search.best_score_
    )

In [ ]:
#@title 5.4. Run Reference 80:20 Model

# ===================================================
# CREATE REFERENCE SPLIT
# ===================================================

reference_train_idx, reference_test_idx = (
    create_train_test_indices(
        target=y,
        test_size=TEST_SIZE,
        random_state=42,
        use_quantiles=USE_QUANTILE_SPLIT,
        n_quantiles=N_QUANTILES
    )
)

X_reference_train_full = (
    feature_df
    .iloc[reference_train_idx]
    .copy()
)

X_reference_test_full = (
    feature_df
    .iloc[reference_test_idx]
    .copy()
)

y_reference_train_original = (
    y.iloc[reference_train_idx]
    .copy()
)

y_reference_test_original = (
    y.iloc[reference_test_idx]
    .copy()
)


# ===================================================
# MODEL TARGET
# ===================================================

y_reference_train_model = transform_target(
    y_reference_train_original,
    use_log=USE_LOG_TARGET
)


# ===================================================
# TRAINING-ONLY FEATURE SELECTION
# ===================================================

(
    X_reference_train,
    reference_rf_selector,
    reference_rf_feature_summary
) = select_features_from_training(
    X_train=X_reference_train_full,
    y_train=y_reference_train_model
)


X_reference_test = transform_unseen_features(
    X_new=X_reference_test_full,
    selector=reference_rf_selector
)


# ===================================================
# TRAINING-ONLY RF OPTIMIZATION
# ===================================================

(
    reference_rf,
    reference_best_params,
    reference_cv_score
) = optimize_random_forest(
    X_train=X_reference_train,
    y_train=y_reference_train_model,
    random_state=42
)


# ===================================================
# PREDICT UNSEEN TEST SET
# ===================================================

reference_pred_model = (
    reference_rf.predict(
        X_reference_test
    )
)

reference_pred_original = (
    inverse_target(
        reference_pred_model,
        use_log=USE_LOG_TARGET
    )
)


# ===================================================
# EVALUATE ON ORIGINAL SCALE
# ===================================================

TARGET_RANGE = (
    y.max()
    - y.min()
)

reference_metrics = (
    calculate_regression_metrics(
        observed=y_reference_test_original,
        predicted=reference_pred_original,
        n_predictors=X_reference_test.shape[1],
        normalization_range=TARGET_RANGE
    )
)


print("Reference model")
print("---------------")
print(
    "Selected features:",
    X_reference_train.shape[1]
)

print(
    "Best RF parameters:",
    reference_best_params
)

print(
    f"Training CV R²: "
    f"{reference_cv_score:.3f}"
)

print(
    f"Test R²: "
    f"{reference_metrics['R2']:.3f}"
)

print(
    f"Test RMSE: "
    f"{reference_metrics['RMSE']:.3f} "
    f"{TARGET_UNIT}"
)

print(
    f"Test NRMSE_range: "
    f"{reference_metrics['NRMSE_Range_Percent']:.2f}%"
)

print(
    f"Test MAE: "
    f"{reference_metrics['MAE']:.3f} "
    f"{TARGET_UNIT}"
)

In [ ]:
#@title 5.5. Run Leakage-Free Monte Carlo Evaluation

monte_carlo_metrics = []
monte_carlo_predictions = []
monte_carlo_features = []
monte_carlo_parameters = []


for simulation in range(
    N_SIMULATIONS
):

    print(
        f"\rMonte Carlo run "
        f"{simulation + 1}/"
        f"{N_SIMULATIONS}",
        end=""
    )

    # ===================================================
    # 1. TRAIN / TEST SPLIT
    # ===================================================

    train_idx, test_idx = (
        create_train_test_indices(
            target=y,
            test_size=TEST_SIZE,
            random_state=simulation,
            use_quantiles=USE_QUANTILE_SPLIT,
            n_quantiles=N_QUANTILES
        )
    )

    X_train_full = (
        feature_df
        .iloc[train_idx]
        .copy()
    )

    X_test_full = (
        feature_df
        .iloc[test_idx]
        .copy()
    )

    y_train_original = (
        y.iloc[train_idx]
        .copy()
    )

    y_test_original = (
        y.iloc[test_idx]
        .copy()
    )

    # ===================================================
    # 2. TRANSFORM TRAINING TARGET
    # ===================================================

    y_train_model = (
        transform_target(
            y_train_original,
            use_log=USE_LOG_TARGET
        )
    )

    # ===================================================
    # 3. TRAINING-ONLY FEATURE SELECTION
    # ===================================================

    (
        X_train_selected,
        selector,
        selected_summary
    ) = select_features_from_training(
        X_train=X_train_full,
        y_train=y_train_model
    )

    X_test_selected = (
        transform_unseen_features(
            X_new=X_test_full,
            selector=selector
        )
    )

    # ===================================================
    # 4. TRAINING-ONLY RF OPTIMIZATION
    # ===================================================

    (
        rf_model,
        best_params,
        best_cv_score
    ) = optimize_random_forest(
        X_train=X_train_selected,
        y_train=y_train_model,
        random_state=simulation
    )

    # ===================================================
    # 5. PREDICT WITHHELD OBSERVATIONS
    # ===================================================

    y_pred_model = (
        rf_model.predict(
            X_test_selected
        )
    )

    y_pred_original = (
        inverse_target(
            y_pred_model,
            use_log=USE_LOG_TARGET
        )
    )

    # ===================================================
    # 6. TEST-SET PERFORMANCE
    # ===================================================

    metrics = (
        calculate_regression_metrics(
            observed=y_test_original,
            predicted=y_pred_original,
            n_predictors=X_test_selected.shape[1],
            normalization_range=TARGET_RANGE
        )
    )

    metrics[
        "Simulation"
    ] = simulation + 1

    metrics[
        "Training_CV_R2"
    ] = best_cv_score

    monte_carlo_metrics.append(
        metrics
    )

    # ===================================================
    # 7. STORE TEST PREDICTIONS
    # ===================================================

    prediction_df = pd.DataFrame({
        "Simulation":
            simulation + 1,

        "Original_Row_Index":
            data.iloc[
                test_idx
            ][
                "original_row_index"
            ].values,

        "Observed":
            y_test_original.values,

        "Predicted":
            y_pred_original
    })

    prediction_df[
        "Residual"
    ] = (
        prediction_df[
            "Predicted"
        ]
        - prediction_df[
            "Observed"
        ]
    )

    monte_carlo_predictions.append(
        prediction_df
    )

    # ===================================================
    # 8. STORE SELECTED FEATURES
    # ===================================================

    for feature in selector[
        "selected_features"
    ]:

        monte_carlo_features.append({
            "Simulation":
                simulation + 1,

            "Feature":
                feature
        })

    # ===================================================
    # 9. STORE RF PARAMETERS
    # ===================================================

    parameter_record = {
        "Simulation":
            simulation + 1,

        "Training_CV_R2":
            best_cv_score
    }

    parameter_record.update(
        best_params
    )

    monte_carlo_parameters.append(
        parameter_record
    )


print(
    "\nMonte Carlo evaluation complete."
)

In [ ]:
#@title 5.6. Summarize Monte Carlo Performance

mc_metrics_df = pd.DataFrame(
    monte_carlo_metrics
)

mc_predictions_df = pd.concat(
    monte_carlo_predictions,
    ignore_index=True
)

mc_features_df = pd.DataFrame(
    monte_carlo_features
)

mc_parameters_df = pd.DataFrame(
    monte_carlo_parameters
)


# ===================================================
# SUMMARY STATISTICS
# ===================================================

metric_columns = [
    "R2",
    "Adjusted_R2",
    "RMSE",
    "NRMSE_Range_Percent",
    "MAE",
    "Bias",
    "Slope",
    "N_Features",
    "Training_CV_R2"
]

summary_records = []

for metric in metric_columns:

    values = (
        pd.to_numeric(
            mc_metrics_df[metric],
            errors="coerce"
        )
        .dropna()
    )

    summary_records.append({
        "Metric": metric,
        "Mean": values.mean(),
        "Std": values.std(ddof=1),
        "Median": values.median(),
        "Minimum": values.min(),
        "Maximum": values.max()
    })


mc_summary_df = pd.DataFrame(
    summary_records
)

display(
    mc_summary_df
)


# ===================================================
# FEATURE-SELECTION FREQUENCY
# ===================================================

feature_frequency_df = (
    mc_features_df[
        "Feature"
    ]
    .value_counts()
    .rename_axis(
        "Feature"
    )
    .reset_index(
        name="Times_Selected"
    )
)

feature_frequency_df[
    "Selection_Frequency_Percent"
] = (
    feature_frequency_df[
        "Times_Selected"
    ]
    / N_SIMULATIONS
    * 100
)

display(
    feature_frequency_df.head(20)
)

In [ ]:
#@title 5.7. Summarize Prediction Uncertainty by Observation

observation_summary_df = (
    mc_predictions_df
    .groupby(
        "Original_Row_Index"
    )
    .agg(
        Observed=(
            "Observed",
            "first"
        ),

        Mean_Predicted=(
            "Predicted",
            "mean"
        ),

        Prediction_SD=(
            "Predicted",
            "std"
        ),

        Prediction_Count=(
            "Predicted",
            "count"
        ),

        Mean_Residual=(
            "Residual",
            "mean"
        )
    )
    .reset_index()
)


display(
    observation_summary_df.head()
)

In [ ]:
#@title 5.8. Export Monte Carlo Results

MC_RESULTS_FILE = os.path.join(
    MODEL_OUTPUT_DIR,
    f"{SCENARIO_NAME}_MonteCarlo_Results.xlsx"
)


with pd.ExcelWriter(
    MC_RESULTS_FILE,
    engine="openpyxl"
) as writer:

    mc_metrics_df.to_excel(
        writer,
        sheet_name="Run_Metrics",
        index=False
    )

    mc_summary_df.to_excel(
        writer,
        sheet_name="Metric_Summary",
        index=False
    )

    mc_predictions_df.to_excel(
        writer,
        sheet_name="Predictions",
        index=False
    )

    observation_summary_df.to_excel(
        writer,
        sheet_name="Observation_Summary",
        index=False
    )

    feature_frequency_df.to_excel(
        writer,
        sheet_name="Feature_Frequency",
        index=False
    )

    mc_parameters_df.to_excel(
        writer,
        sheet_name="RF_Parameters",
        index=False
    )


print(
    f"Monte Carlo results saved to:\n"
    f"{MC_RESULTS_FILE}"
)

In [ ]:
#@title 5.9. Train Final Model for Spatial Prediction

# ===================================================
# MODEL TARGET
# ===================================================

y_full_model = transform_target(
    y,
    use_log=USE_LOG_TARGET
)


# ===================================================
# FEATURE SELECTION USING ALL AVAILABLE DATA
# ===================================================

(
    X_final_model,
    final_selector,
    final_feature_summary
) = select_features_from_training(
    X_train=feature_df,
    y_train=y_full_model
)


# ===================================================
# RF OPTIMIZATION USING ALL AVAILABLE DATA
# ===================================================

(
    final_rf_model,
    final_rf_params,
    final_cv_score
) = optimize_random_forest(
    X_train=X_final_model,
    y_train=y_full_model,
    random_state=42
)


# ===================================================
# FINAL MODEL SUMMARY
# ===================================================

FINAL_SELECTED_FEATURES = (
    final_selector[
        "selected_features"
    ]
)

print(
    "Final model trained."
)

print(
    f"Selected features: "
    f"{len(FINAL_SELECTED_FEATURES)}"
)

print(
    f"Cross-validated training R²: "
    f"{final_cv_score:.3f}"
)

print(
    "\nFinal RF parameters:"
)

print(
    final_rf_params
)

print(
    "\nFinal selected features:"
)

for feature in FINAL_SELECTED_FEATURES:
    print(
        " -",
        feature
    )

In [ ]:
#@title 5.10. Export Final Model Configuration

FINAL_MODEL_CONFIG_FILE = os.path.join(
    MODEL_OUTPUT_DIR,
    f"{SCENARIO_NAME}_FinalModel_Config.xlsx"
)


final_parameters_df = pd.DataFrame(
    list(
        final_rf_params.items()
    ),
    columns=[
        "Parameter",
        "Value"
    ]
)


with pd.ExcelWriter(
    FINAL_MODEL_CONFIG_FILE,
    engine="openpyxl"
) as writer:

    final_feature_summary.to_excel(
        writer,
        sheet_name="Selected_Features",
        index=False
    )

    final_parameters_df.to_excel(
        writer,
        sheet_name="RF_Parameters",
        index=False
    )

    final_selector[
        "selection_log"
    ].to_excel(
        writer,
        sheet_name="Feature_Selection_Log",
        index=False
    )

    pd.DataFrame({
        "Setting": [
            "DATA_SOURCE",
            "WINDOW_SIZE",
            "TARGET",
            "USE_LOG_TARGET",
            "N_OBSERVATIONS",
            "N_SELECTED_FEATURES",
            "FINAL_CV_R2"
        ],

        "Value": [
            DATA_SOURCE,
            WINDOW_SIZE,
            TARGET,
            USE_LOG_TARGET,
            len(y),
            len(
                FINAL_SELECTED_FEATURES
            ),
            final_cv_score
        ]
    }).to_excel(
        writer,
        sheet_name="Model_Settings",
        index=False
    )


print(
    f"Final model configuration saved to:\n"
    f"{FINAL_MODEL_CONFIG_FILE}"
)

In [ ]:
#@title 🌍 6. SPATIO-TEMPORAL WATER QUALITY MAPPING

This section applies the final Random Forest model to Landsat imagery to generate spatially continuous water-quality predictions.

Landsat 8 and Landsat 9 imagery is retrieved for the selected lake and mapping period. Cloud and cloud-shadow pixels are masked using the same QA criteria applied during spectral extraction. TOA or Surface Reflectance imagery is prepared according to the model configuration and exported as seven-band GeoTIFF files.

The final spectral features selected during model development are then generated for every valid image pixel using the same feature-engineering function used during model training. The trained Random Forest model is applied to these predictors, and predictions are converted back to the original target scale when a log-transformed model is used.

Individual prediction maps are retained for subsequent monthly and multi-year spatio-temporal analysis.

In [ ]:
#@title 6.1. Configure Spatio-Temporal Mapping

# ===================================================
# LAKE
# ===================================================

LAKE_NAME = "Conesus"

# Folder containing one boundary shapefile per lake.
LAKE_BOUNDARY_DIR = os.path.join(
    INPUT_DIR,
    "lake_boundaries"
)

# Expected structure:
# lake_boundaries/
# ├── Conesus.shp
# ├── Seneca.shp
# ├── Cayuga.shp
# └── ...

LAKE_BOUNDARY_FILE = os.path.join(
    LAKE_BOUNDARY_DIR,
    f"{LAKE_NAME}.shp"
)

# ===================================================
# MAPPING PERIOD
# ===================================================

MAP_START_YEAR = 2017
MAP_END_YEAR = 2025

# Mapping period can differ from field-matchup period.
# Default here covers April-November.
MAP_START_MONTH = 4
MAP_START_DAY = 1

MAP_END_MONTH = 11
MAP_END_DAY = 30

# ===================================================
# LANDSAT PRODUCT
# ===================================================

# Use the same product used by the final model.
MAP_DATA_SOURCE = DATA_SOURCE

# ===================================================
# CLOUD FILTER
# ===================================================

# Maximum lake-specific cloudy fraction.
MAX_LAKE_CLOUD_FRACTION = 0.10

# ===================================================
# OUTPUTS
# ===================================================

MAP_OUTPUT_DIR = os.path.join(
    OUTPUT_DIR,
    "maps",
    LAKE_NAME,
    MAP_DATA_SOURCE,
    TARGET
)

LANDSAT_EXPORT_FOLDER = (
    f"GEE_{LAKE_NAME}_{MAP_DATA_SOURCE}"
)

os.makedirs(
    MAP_OUTPUT_DIR,
    exist_ok=True
)

# ===================================================
# PREDICTION SETTINGS
# ===================================================

# Process pixels in smaller batches to limit memory use.
PREDICTION_CHUNK_SIZE = 2000

print("Mapping configuration")
print("---------------------")
print(f"Lake          : {LAKE_NAME}")
print(f"Product       : {MAP_DATA_SOURCE}")
print(f"Target        : {TARGET_LABEL}")
print(f"Period        : {MAP_START_YEAR}-{MAP_END_YEAR}")
print(
    f"Season        : "
    f"{MAP_START_MONTH:02d}-{MAP_START_DAY:02d} "
    f"to "
    f"{MAP_END_MONTH:02d}-{MAP_END_DAY:02d}"
)
print(
    f"Cloud limit   : "
    f"{MAX_LAKE_CLOUD_FRACTION:.0%}"
)

In [ ]:
#@title 6.2. Load Lake Boundary

if not os.path.exists(
    LAKE_BOUNDARY_FILE
):
    raise FileNotFoundError(
        f"Lake boundary not found:\n"
        f"{LAKE_BOUNDARY_FILE}"
    )

lake_gdf = gpd.read_file(
    LAKE_BOUNDARY_FILE
)

lake_gdf = lake_gdf.to_crs(
    epsg=4326
)

lake_geojson = (
    lake_gdf.__geo_interface__
)

lake_fc = ee.FeatureCollection(
    lake_geojson
)

roi = lake_fc.geometry()

print(
    f"{LAKE_NAME} boundary loaded."
)

print(
    f"CRS: {lake_gdf.crs}"
)

In [ ]:
#@title 6.3. Prepare Landsat Collection for Mapping

MAP_TOA_BANDS = [
    "B1", "B2", "B3", "B4",
    "B5", "B6", "B7"
]

MAP_SR_BANDS = [
    "SR_B1", "SR_B2", "SR_B3", "SR_B4",
    "SR_B5", "SR_B6", "SR_B7"
]


def mapping_clear_mask(image):
    """
    Mask Landsat cloud and cloud-shadow pixels
    using QA_PIXEL.
    """

    qa = image.select(
        "QA_PIXEL"
    )

    cloud = (
        qa.bitwiseAnd(
            1 << 3
        ).eq(0)
    )

    shadow = (
        qa.bitwiseAnd(
            1 << 4
        ).eq(0)
    )

    return cloud.And(
        shadow
    )


def prepare_mapping_image(image):
    """
    Prepare Landsat bands using the same product
    convention used during model development.
    """

    clear_mask = (
        mapping_clear_mask(
            image
        )
    )

    if MAP_DATA_SOURCE == "TOA":

        spectral = (
            image
            .select(
                MAP_TOA_BANDS
            )
            .updateMask(
                clear_mask
            )
            .toFloat()
        )

    elif MAP_DATA_SOURCE == "SR":

        spectral = (
            image
            .select(
                MAP_SR_BANDS
            )
            .multiply(
                0.0000275
            )
            .add(
                -0.2
            )
            .rename(
                MAP_TOA_BANDS
            )
            .updateMask(
                clear_mask
            )
            .toFloat()
        )

    else:

        raise ValueError(
            "MAP_DATA_SOURCE must "
            "be 'TOA' or 'SR'."
        )

    return (
        spectral
        .copyProperties(
            image,
            image.propertyNames()
        )
    )

In [ ]:
#@title 6.4. Calculate Lake-Specific Cloud Fraction

def add_lake_cloud_fraction(
    image,
    geometry
):
    """
    Calculate the fraction of cloud and
    cloud-shadow pixels within the lake ROI.
    """

    qa = image.select(
        "QA_PIXEL"
    )

    cloud = (
        qa.bitwiseAnd(
            1 << 3
        ).neq(0)
    )

    shadow = (
        qa.bitwiseAnd(
            1 << 4
        ).neq(0)
    )

    cloudy_pixels = (
        cloud.Or(
            shadow
        )
    )

    cloud_fraction = (
        cloudy_pixels
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=geometry,
            scale=30,
            maxPixels=1e13
        )
        .get(
            "QA_PIXEL"
        )
    )

    return image.set(
        "lake_cloud_fraction",
        cloud_fraction
    )

In [ ]:
#@title  6.5. Build Landsat 8 and 9 Mapping Collection

def get_mapping_collection(
    year,
    roi
):
    """
    Retrieve Landsat 8 and Landsat 9 imagery
    for one mapping year.
    """

    start_date = ee.Date.fromYMD(
        year,
        MAP_START_MONTH,
        MAP_START_DAY
    )

    end_date = (
        ee.Date.fromYMD(
            year,
            MAP_END_MONTH,
            MAP_END_DAY
        )
        .advance(
            1,
            "day"
        )
    )

    if MAP_DATA_SOURCE == "TOA":

        dataset_l8 = (
            "LANDSAT/LC08/C02/T1_TOA"
        )

        dataset_l9 = (
            "LANDSAT/LC09/C02/T1_TOA"
        )

    elif MAP_DATA_SOURCE == "SR":

        dataset_l8 = (
            "LANDSAT/LC08/C02/T1_L2"
        )

        dataset_l9 = (
            "LANDSAT/LC09/C02/T1_L2"
        )

    else:

        raise ValueError(
            "MAP_DATA_SOURCE must "
            "be 'TOA' or 'SR'."
        )

    l8 = (
        ee.ImageCollection(
            dataset_l8
        )
        .filterBounds(
            roi
        )
        .filterDate(
            start_date,
            end_date
        )
    )

    l9 = (
        ee.ImageCollection(
            dataset_l9
        )
        .filterBounds(
            roi
        )
        .filterDate(
            start_date,
            end_date
        )
    )

    collection = (
        l8
        .merge(l9)
        .map(
            lambda image:
                add_lake_cloud_fraction(
                    image,
                    roi
                )
        )
        .filter(
            ee.Filter.lt(
                "lake_cloud_fraction",
                MAX_LAKE_CLOUD_FRACTION
            )
        )
        .map(
            prepare_mapping_image
        )
        .sort(
            "system:time_start"
        )
    )

    return collection

In [ ]:
#@title 6.6. Export Mapping Imagery to Google Drive

def export_mapping_year(
    year
):
    """
    Export valid Landsat images for one year.
    """

    collection = (
        get_mapping_collection(
            year=year,
            roi=roi
        )
    )

    image_count = (
        collection
        .size()
        .getInfo()
    )

    print(
        f"\n{year}: "
        f"{image_count} image(s)"
    )

    if image_count == 0:
        return

    image_list = (
        collection
        .toList(
            image_count
        )
    )

    for image_index in range(
        image_count
    ):

        image = ee.Image(
            image_list.get(
                image_index
            )
        )

        image_date = (
            ee.Date(
                image.get(
                    "system:time_start"
                )
            )
            .format(
                "YYYY-MM-dd"
            )
            .getInfo()
        )

        # -------------------------------------------
        # CHECK THAT VALID PIXELS EXIST
        # -------------------------------------------

        valid_count = (
            image
            .select("B1")
            .reduceRegion(
                reducer=ee.Reducer.count(),
                geometry=roi,
                scale=30,
                maxPixels=1e13
            )
            .get("B1")
            .getInfo()
        )

        if (
            valid_count is None
            or valid_count == 0
        ):

            print(
                f"Skipping empty image: "
                f"{image_date}"
            )

            continue

        file_prefix = (
            f"{LAKE_NAME}_"
            f"{MAP_DATA_SOURCE}_"
            f"{image_date}"
        )

        task = (
            ee.batch.Export.image
            .toDrive(
                image=image.clip(
                    roi
                ),

                description=file_prefix,

                folder=(
                    LANDSAT_EXPORT_FOLDER
                ),

                fileNamePrefix=(
                    file_prefix
                ),

                region=roi.bounds(),

                scale=30,

                maxPixels=1e13
            )
        )

        task.start()

        print(
            f"Started: {file_prefix}"
        )


for year in range(
    MAP_START_YEAR,
    MAP_END_YEAR + 1
):

    export_mapping_year(
        year
    )

In [ ]:
#@title 6.7. Configure Raster Prediction

# Google Drive location created by Earth Engine export.
MAPPING_IMAGE_DIR = os.path.join(
    DRIVE_ROOT,
    LANDSAT_EXPORT_FOLDER
)

# Use the final model created in Section 5.
mapping_model = final_rf_model

mapping_features = list(
    FINAL_SELECTED_FEATURES
)

print(
    f"Model target: {TARGET_LABEL}"
)

print(
    f"Selected features: "
    f"{len(mapping_features)}"
)

for feature in mapping_features:
    print(
        " -",
        feature
    )

print(
    f"\nInput imagery:\n"
    f"{MAPPING_IMAGE_DIR}"
)

print(
    f"\nPrediction output:\n"
    f"{MAP_OUTPUT_DIR}"
)

In [ ]:
#@title 6.8. Read Landsat Raster

RASTER_BANDS = [
    "B1", "B2", "B3", "B4",
    "B5", "B6", "B7"
]


def read_landsat_raster(
    image_path
):
    """
    Read a seven-band Landsat GeoTIFF and
    return valid spectral pixels.
    """

    with rasterio.open(
        image_path
    ) as src:

        raster = (
            src.read()
            .astype(
                "float32"
            )
        )

        profile = (
            src.profile.copy()
        )

    if raster.shape[0] < 7:

        raise ValueError(
            f"{os.path.basename(image_path)} "
            "contains fewer than seven bands."
        )

    raster = raster[
        :7,
        :,
        :
    ]

    n_bands, n_rows, n_cols = (
        raster.shape
    )

    pixels = (
        raster
        .reshape(
            n_bands,
            -1
        )
        .T
    )

    pixel_df = pd.DataFrame(
        pixels,
        columns=RASTER_BANDS
    )

    pixel_df = (
        pixel_df
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
    )

    valid_mask = (
        pixel_df[
            RASTER_BANDS
        ]
        .notna()
        .all(
            axis=1
        )
    )

    valid_pixels = (
        pixel_df.loc[
            valid_mask
        ]
        .copy()
    )

    return (
        valid_pixels,
        valid_mask.values,
        profile,
        n_rows,
        n_cols
    )

In [ ]:
#@title 6.9. Generate Model Features and Predict Raster

def predict_raster(
    image_path
):
    """
    Generate model predictors from Landsat
    pixels and create a water-quality raster.
    """

    print(
        f"\nProcessing: "
        f"{os.path.basename(image_path)}"
    )

    (
        valid_pixels,
        valid_mask,
        profile,
        n_rows,
        n_cols
    ) = read_landsat_raster(
        image_path
    )

    print(
        f"Valid spectral pixels: "
        f"{len(valid_pixels)}"
    )

    if valid_pixels.empty:

        print(
            "No valid pixels. "
            "Skipping image."
        )

        return None

    predictions_all = np.full(
        len(valid_pixels),
        np.nan,
        dtype=np.float32
    )

    # ===================================================
    # PROCESS PIXELS IN CHUNKS
    # ===================================================

    for start in range(
        0,
        len(valid_pixels),
        PREDICTION_CHUNK_SIZE
    ):

        end = min(
            start
            + PREDICTION_CHUNK_SIZE,

            len(valid_pixels)
        )

        chunk = (
            valid_pixels.iloc[
                start:end
            ]
            .copy()
        )

        # -----------------------------------------------
        # USE EXACT SAME FEATURE ENGINEERING AS TRAINING
        # -----------------------------------------------

        chunk_features = (
            generate_spectral_features(
                input_data=chunk,
                bands=RASTER_BANDS
            )
        )

        missing_features = [
            feature
            for feature
            in mapping_features
            if feature
            not in chunk_features.columns
        ]

        if missing_features:

            raise ValueError(
                "The following model features "
                "could not be generated:\n"
                + "\n".join(
                    missing_features
                )
            )

        X_chunk = (
            chunk_features[
                mapping_features
            ]
            .replace(
                [np.inf, -np.inf],
                np.nan
            )
        )

        good_rows = (
            X_chunk
            .notna()
            .all(axis=1)
        )

        if not good_rows.any():
            continue

        model_prediction = (
            mapping_model.predict(
                X_chunk.loc[
                    good_rows
                ]
            )
        )

        prediction_original = (
            inverse_target(
                model_prediction,
                use_log=USE_LOG_TARGET
            )
        )

        good_positions = (
            np.where(
                good_rows.values
            )[0]
        )

        predictions_all[
            start
            + good_positions
        ] = (
            prediction_original
            .astype(
                np.float32
            )
        )

    # ===================================================
    # RECONSTRUCT RASTER
    # ===================================================

    prediction_flat = np.full(
        n_rows * n_cols,
        np.nan,
        dtype=np.float32
    )

    valid_indices = np.where(
        valid_mask
    )[0]

    prediction_flat[
        valid_indices
    ] = predictions_all

    prediction_raster = (
        prediction_flat
        .reshape(
            n_rows,
            n_cols
        )
    )

    # ===================================================
    # OUTPUT PROFILE
    # ===================================================

    profile.update(
        dtype="float32",
        count=1,
        nodata=np.nan
    )

    return (
        prediction_raster,
        profile
    )

In [ ]:
#@title 6.10. Generate Individual Prediction Maps

def extract_image_date(
    filename
):
    """
    Extract YYYY-MM-DD from Landsat filename.
    """

    match = re.search(
        r"\d{4}-\d{2}-\d{2}",
        filename
    )

    if match:
        return match.group(0)

    return "unknown_date"


image_files = sorted(
    glob.glob(
        os.path.join(
            MAPPING_IMAGE_DIR,
            "*.tif"
        )
    )
)

if not image_files:

    raise FileNotFoundError(
        f"No Landsat GeoTIFF files found in:\n"
        f"{MAPPING_IMAGE_DIR}"
    )


print(
    f"Images found: "
    f"{len(image_files)}"
)


prediction_records = []


for image_path in image_files:

    result = predict_raster(
        image_path
    )

    if result is None:
        continue

    prediction_raster, profile = (
        result
    )

    image_date = extract_image_date(
        os.path.basename(
            image_path
        )
    )

    output_name = (
        f"{TARGET}_"
        f"{LAKE_NAME}_"
        f"{image_date}.tif"
    )

    output_path = os.path.join(
        MAP_OUTPUT_DIR,
        output_name
    )

    with rasterio.open(
        output_path,
        "w",
        **profile
    ) as dst:

        dst.write(
            prediction_raster,
            1
        )

    valid_predictions = (
        prediction_raster[
            np.isfinite(
                prediction_raster
            )
        ]
    )

    prediction_records.append({
        "Lake":
            LAKE_NAME,

        "Date":
            image_date,

        "Image_File":
            os.path.basename(
                output_path
            ),

        "Valid_Pixels":
            len(
                valid_predictions
            ),

        "Mean":
            (
                np.mean(
                    valid_predictions
                )
                if len(
                    valid_predictions
                )
                else np.nan
            ),

        "Median":
            (
                np.median(
                    valid_predictions
                )
                if len(
                    valid_predictions
                )
                else np.nan
            ),

        "Std":
            (
                np.std(
                    valid_predictions
                )
                if len(
                    valid_predictions
                )
                else np.nan
            )
    })

    print(
        f"Saved: {output_name}"
    )


prediction_summary_df = (
    pd.DataFrame(
        prediction_records
    )
)

prediction_summary_df[
    "Date"
] = pd.to_datetime(
    prediction_summary_df[
        "Date"
    ],
    errors="coerce"
)

display(
    prediction_summary_df.head()
)

In [ ]:
#@title 6.11. Generate Monthly Prediction Summary

prediction_summary_df[
    "Year"
] = (
    prediction_summary_df[
        "Date"
    ].dt.year
)

prediction_summary_df[
    "Month"
] = (
    prediction_summary_df[
        "Date"
    ].dt.month
)


monthly_prediction_summary = (
    prediction_summary_df
    .groupby(
        [
            "Lake",
            "Year",
            "Month"
        ],
        as_index=False
    )
    .agg(
        Monthly_Mean=(
            "Mean",
            "mean"
        ),

        Monthly_Median=(
            "Median",
            "median"
        ),

        Monthly_Std=(
            "Mean",
            "std"
        ),

        Image_Count=(
            "Image_File",
            "count"
        )
    )
)


monthly_prediction_summary[
    "Date"
] = pd.to_datetime(
    dict(
        year=monthly_prediction_summary[
            "Year"
        ],

        month=monthly_prediction_summary[
            "Month"
        ],

        day=1
    )
)


monthly_prediction_summary = (
    monthly_prediction_summary
    .sort_values(
        "Date"
    )
    .reset_index(
        drop=True
    )
)


display(
    monthly_prediction_summary.head()
)

In [ ]:
#@title 6.12. Export Mapping Summary

MAP_SUMMARY_FILE = os.path.join(
    MAP_OUTPUT_DIR,
    (
        f"{LAKE_NAME}_"
        f"{MAP_DATA_SOURCE}_"
        f"{TARGET}_"
        f"Mapping_Summary.xlsx"
    )
)


with pd.ExcelWriter(
    MAP_SUMMARY_FILE,
    engine="openpyxl"
) as writer:

    prediction_summary_df.to_excel(
        writer,
        sheet_name="Image_Statistics",
        index=False
    )

    monthly_prediction_summary.to_excel(
        writer,
        sheet_name="Monthly_Summary",
        index=False
    )


print(
    f"Mapping summary saved to:\n"
    f"{MAP_SUMMARY_FILE}"
)

In [ ]:
#@title 🗺️ 7. VISUALIZATION AND SUMMARY PRODUCTS

This section summarizes and visualizes the generated water-quality predictions. Individual prediction rasters can be displayed using a common scale for temporal comparison, while image-level statistics are combined across lakes to examine monthly and multi-year patterns.

Monthly heatmaps summarize predicted water-quality conditions and corresponding Landsat image availability across lakes. Long-term pixel-wise median and coefficient-of-variation maps are also generated to characterize persistent spatial patterns and temporal variability.

These visualization functions are independent of model fitting and can be adapted according to the requirements of individual applications.

In [ ]:
#@title 7.1. Configure Visualization

# ===================================================
# VISUALIZATION SETTINGS
# ===================================================

VIS_DATA_SOURCE = DATA_SOURCE
VIS_TARGET = TARGET

VIS_START_YEAR = 2017
VIS_END_YEAR = 2025

VIS_MONTHS = [
    4, 5, 6, 7,
    8, 9, 10, 11
]

# Consistent lake ordering
VIS_LAKE_ORDER = [
    "Otisco",
    "Skaneateles",
    "Owasco",
    "Cayuga",
    "Seneca",
    "Keuka",
    "Canandaigua",
    "Honeoye",
    "Canadice",
    "Hemlock",
    "Conesus"
]

VIS_OUTPUT_DIR = os.path.join(
    OUTPUT_DIR,
    "visualization",
    VIS_DATA_SOURCE,
    VIS_TARGET
)

os.makedirs(
    VIS_OUTPUT_DIR,
    exist_ok=True
)

if VIS_TARGET == "sdd":
    VIS_LABEL = "SDD"
    VIS_UNIT = "m"
    VIS_CMAP = "Blues_r"

elif VIS_TARGET == "chl_a":
    VIS_LABEL = "Chl-a"
    VIS_UNIT = "µg/L"
    VIS_CMAP = "YlOrRd"

else:
    raise ValueError(
        "VIS_TARGET must be 'sdd' or 'chl_a'."
    )

print("Visualization configuration")
print("---------------------------")
print(f"Product : {VIS_DATA_SOURCE}")
print(f"Target  : {VIS_LABEL}")
print(f"Period  : {VIS_START_YEAR}-{VIS_END_YEAR}")

In [ ]:
#@title 7.2. Display Spatio-Temporal Prediction Maps

def plot_prediction_maps(
    map_directory,
    target_label=VIS_LABEL,
    target_unit=VIS_UNIT,
    cmap=VIS_CMAP,
    columns=4,
    lower_percentile=5,
    upper_percentile=95
):
    """
    Plot all prediction rasters in one directory
    using a common visualization range.
    """

    raster_files = sorted(
        glob.glob(
            os.path.join(
                map_directory,
                "*.tif"
            )
        )
    )

    if not raster_files:
        raise FileNotFoundError(
            f"No prediction rasters found in:\n"
            f"{map_directory}"
        )

    arrays = []
    valid_values = []

    # ===================================================
    # LOAD RASTERS
    # ===================================================

    for raster_file in raster_files:

        with rasterio.open(
            raster_file
        ) as src:

            array = (
                src.read(1)
                .astype("float32")
            )

            nodata = src.nodata

        if nodata is not None:
            array[
                array == nodata
            ] = np.nan

        array[
            ~np.isfinite(array)
        ] = np.nan

        arrays.append(
            array
        )

        values = array[
            np.isfinite(array)
        ]

        if len(values):
            valid_values.append(
                values
            )

    all_values = np.concatenate(
        valid_values
    )

    # ===================================================
    # COMMON DISPLAY RANGE
    # ===================================================

    vmin = np.nanpercentile(
        all_values,
        lower_percentile
    )

    vmax = np.nanpercentile(
        all_values,
        upper_percentile
    )

    print(
        f"Display range: "
        f"{vmin:.2f} – {vmax:.2f} "
        f"{target_unit}"
    )

    # ===================================================
    # PLOT GRID
    # ===================================================

    rows = int(
        np.ceil(
            len(arrays)
            / columns
        )
    )

    fig, axes = plt.subplots(
        rows,
        columns,
        figsize=(
            4 * columns,
            4 * rows
        )
    )

    axes = np.atleast_1d(
        axes
    ).flatten()

    for index, (
        array,
        raster_file
    ) in enumerate(
        zip(
            arrays,
            raster_files
        )
    ):

        image = axes[index].imshow(
            np.clip(
                array,
                vmin,
                vmax
            ),
            cmap=cmap,
            vmin=vmin,
            vmax=vmax
        )

        date = extract_image_date(
            os.path.basename(
                raster_file
            )
        )

        axes[index].set_title(
            date,
            fontsize=9
        )

        axes[index].axis(
            "off"
        )

    for index in range(
        len(arrays),
        len(axes)
    ):
        axes[index].axis(
            "off"
        )

    fig.subplots_adjust(
        right=0.90
    )

    colorbar_axis = fig.add_axes(
        [
            0.92,
            0.15,
            0.02,
            0.70
        ]
    )

    colorbar = fig.colorbar(
        image,
        cax=colorbar_axis
    )

    colorbar.set_label(
        f"Predicted {target_label} "
        f"({target_unit})"
    )

    plt.show()


plot_prediction_maps(
    map_directory=MAP_OUTPUT_DIR
)

In [ ]:
#@title 7.3. Combine Prediction Summaries Across Lakes

def collect_lake_mapping_summaries(
    base_map_directory,
    lake_names,
    data_source,
    target
):
    """
    Combine mapping-summary workbooks produced
    for multiple lakes.
    """

    image_summaries = []
    monthly_summaries = []

    for lake_name in lake_names:

        lake_directory = os.path.join(
            base_map_directory,
            lake_name,
            data_source,
            target
        )

        summary_file = os.path.join(
            lake_directory,
            (
                f"{lake_name}_"
                f"{data_source}_"
                f"{target}_"
                f"Mapping_Summary.xlsx"
            )
        )

        if not os.path.exists(
            summary_file
        ):

            print(
                f"Missing summary: "
                f"{lake_name}"
            )

            continue

        image_df = pd.read_excel(
            summary_file,
            sheet_name="Image_Statistics"
        )

        monthly_df = pd.read_excel(
            summary_file,
            sheet_name="Monthly_Summary"
        )

        image_summaries.append(
            image_df
        )

        monthly_summaries.append(
            monthly_df
        )

    if not image_summaries:

        raise ValueError(
            "No mapping summaries were found."
        )

    combined_images = pd.concat(
        image_summaries,
        ignore_index=True
    )

    combined_monthly = pd.concat(
        monthly_summaries,
        ignore_index=True
    )

    return (
        combined_images,
        combined_monthly
    )


(
    all_image_summary_df,
    all_monthly_summary_df
) = collect_lake_mapping_summaries(
    base_map_directory=os.path.join(
        OUTPUT_DIR,
        "maps"
    ),
    lake_names=VIS_LAKE_ORDER,
    data_source=VIS_DATA_SOURCE,
    target=VIS_TARGET
)


print(
    f"Images summarized: "
    f"{len(all_image_summary_df)}"
)

print(
    f"Lakes represented: "
    f"{all_image_summary_df['Lake'].nunique()}"
)

In [ ]:
#@title 7.4. Create Cross-Lake Monthly Summary

all_monthly_summary_df[
    "Date"
] = pd.to_datetime(
    all_monthly_summary_df[
        "Date"
    ],
    errors="coerce"
)

all_monthly_summary_df = (
    all_monthly_summary_df
    .dropna(
        subset=["Date"]
    )
)

all_monthly_summary_df = (
    all_monthly_summary_df[
        (
            all_monthly_summary_df[
                "Date"
            ].dt.year
            >= VIS_START_YEAR
        )
        &
        (
            all_monthly_summary_df[
                "Date"
            ].dt.year
            <= VIS_END_YEAR
        )
        &
        (
            all_monthly_summary_df[
                "Date"
            ].dt.month
            .isin(
                VIS_MONTHS
            )
        )
    ]
    .copy()
)


# ===================================================
# MONTHLY MEAN MATRIX
# ===================================================

monthly_mean_matrix = (
    all_monthly_summary_df
    .pivot(
        index="Lake",
        columns="Date",
        values="Monthly_Mean"
    )
    .reindex(
        VIS_LAKE_ORDER
    )
)


# ===================================================
# IMAGE COUNT MATRIX
# ===================================================

monthly_count_matrix = (
    all_monthly_summary_df
    .pivot(
        index="Lake",
        columns="Date",
        values="Image_Count"
    )
    .reindex(
        VIS_LAKE_ORDER
    )
)


# Ensure identical columns
monthly_count_matrix = (
    monthly_count_matrix
    .reindex(
        columns=monthly_mean_matrix.columns
    )
)


display(
    monthly_mean_matrix.head()
)

In [ ]:
#@title 7.5. Plot Monthly Spatio-Temporal Heatmap

fig, ax = plt.subplots(
    figsize=(22, 8)
)

sns.heatmap(
    monthly_mean_matrix,
    cmap=VIS_CMAP,
    linewidths=0.3,
    linecolor="white",
    ax=ax,
    cbar_kws={
        "label":
            f"Predicted {VIS_LABEL} "
            f"({VIS_UNIT})"
    }
)

ax.set_xticklabels(
    [
        pd.Timestamp(date)
        .strftime("%b %Y")
        for date
        in monthly_mean_matrix.columns
    ],
    rotation=90,
    fontsize=9
)

ax.set_yticklabels(
    ax.get_yticklabels(),
    rotation=0,
    fontsize=11
)

ax.set_xlabel(
    "Month"
)

ax.set_ylabel(
    "Lake"
)

ax.set_title(
    (
        f"Monthly Mean Predicted "
        f"{VIS_LABEL}: "
        f"{VIS_START_YEAR}–"
        f"{VIS_END_YEAR}"
    )
)

plt.tight_layout()


MONTHLY_HEATMAP_FILE = os.path.join(
    VIS_OUTPUT_DIR,
    (
        f"{VIS_TARGET}_"
        f"{VIS_DATA_SOURCE}_"
        f"Monthly_Heatmap.png"
    )
)

plt.savefig(
    MONTHLY_HEATMAP_FILE,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    f"Saved to:\n"
    f"{MONTHLY_HEATMAP_FILE}"
)

In [ ]:
#@title 7.6. Calculate Long-Term Pixel-Wise Statistics

def calculate_long_term_raster_statistics(
    map_directory
):
    """
    Calculate pixel-wise long-term median,
    mean, standard deviation and CV from
    prediction rasters.
    """

    raster_files = sorted(
        glob.glob(
            os.path.join(
                map_directory,
                "*.tif"
            )
        )
    )

    if not raster_files:

        raise FileNotFoundError(
            f"No prediction rasters found in:\n"
            f"{map_directory}"
        )

    arrays = []
    profile = None

    for raster_file in raster_files:

        with rasterio.open(
            raster_file
        ) as src:

            array = (
                src.read(1)
                .astype(
                    "float32"
                )
            )

            if profile is None:
                profile = (
                    src.profile.copy()
                )

            nodata = src.nodata

        if nodata is not None:

            array[
                array == nodata
            ] = np.nan

        array[
            ~np.isfinite(array)
        ] = np.nan

        arrays.append(
            array
        )

    raster_stack = np.stack(
        arrays,
        axis=0
    )

    with warnings.catch_warnings():

        warnings.simplefilter(
            "ignore",
            category=RuntimeWarning
        )

        pixel_mean = np.nanmean(
            raster_stack,
            axis=0
        )

        pixel_median = np.nanmedian(
            raster_stack,
            axis=0
        )

        pixel_std = np.nanstd(
            raster_stack,
            axis=0
        )

    pixel_cv = np.where(
        np.isfinite(pixel_mean)
        &
        (np.abs(pixel_mean) > 0),

        pixel_std
        / np.abs(pixel_mean)
        * 100,

        np.nan
    )

    return {
        "mean": pixel_mean,
        "median": pixel_median,
        "std": pixel_std,
        "cv": pixel_cv,
        "profile": profile,
        "n_images": len(raster_files)
    }


long_term_stats = (
    calculate_long_term_raster_statistics(
        MAP_OUTPUT_DIR
    )
)

print(
    f"Images included: "
    f"{long_term_stats['n_images']}"
)

In [ ]:
#@title 7.7. Export Long-Term Summary Rasters

LONG_TERM_OUTPUT_DIR = os.path.join(
    MAP_OUTPUT_DIR,
    "long_term"
)

os.makedirs(
    LONG_TERM_OUTPUT_DIR,
    exist_ok=True
)


def export_single_band_raster(
    array,
    profile,
    output_path
):

    output_profile = (
        profile.copy()
    )

    output_profile.update(
        dtype="float32",
        count=1,
        nodata=np.nan
    )

    with rasterio.open(
        output_path,
        "w",
        **output_profile
    ) as dst:

        dst.write(
            array.astype(
                "float32"
            ),
            1
        )


MEDIAN_MAP_FILE = os.path.join(
    LONG_TERM_OUTPUT_DIR,
    (
        f"{LAKE_NAME}_"
        f"{VIS_TARGET}_"
        f"LongTerm_Median.tif"
    )
)

CV_MAP_FILE = os.path.join(
    LONG_TERM_OUTPUT_DIR,
    (
        f"{LAKE_NAME}_"
        f"{VIS_TARGET}_"
        f"LongTerm_CV.tif"
    )
)


export_single_band_raster(
    long_term_stats[
        "median"
    ],
    long_term_stats[
        "profile"
    ],
    MEDIAN_MAP_FILE
)

export_single_band_raster(
    long_term_stats[
        "cv"
    ],
    long_term_stats[
        "profile"
    ],
    CV_MAP_FILE
)


print(
    f"Median map:\n"
    f"{MEDIAN_MAP_FILE}"
)

print(
    f"\nCV map:\n"
    f"{CV_MAP_FILE}"
)

In [ ]:
#@title 7.8. Visualize Long-Term Spatial Patterns

def plot_long_term_map(
    raster_array,
    title,
    colorbar_label,
    cmap,
    lower_percentile=2,
    upper_percentile=98
):

    valid = raster_array[
        np.isfinite(
            raster_array
        )
    ]

    if not len(valid):
        raise ValueError(
            "Raster contains no valid values."
        )

    vmin = np.percentile(
        valid,
        lower_percentile
    )

    vmax = np.percentile(
        valid,
        upper_percentile
    )

    fig, ax = plt.subplots(
        figsize=(7, 8)
    )

    image = ax.imshow(
        raster_array,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax
    )

    ax.set_title(
        title
    )

    ax.axis(
        "off"
    )

    colorbar = fig.colorbar(
        image,
        ax=ax,
        fraction=0.04,
        pad=0.03
    )

    colorbar.set_label(
        colorbar_label
    )

    plt.tight_layout()
    plt.show()


# ===================================================
# LONG-TERM MEDIAN
# ===================================================

plot_long_term_map(
    raster_array=long_term_stats[
        "median"
    ],

    title=(
        f"{LAKE_NAME}: "
        f"Long-Term Median "
        f"{VIS_LABEL}"
    ),

    colorbar_label=(
        f"Predicted {VIS_LABEL} "
        f"({VIS_UNIT})"
    ),

    cmap=VIS_CMAP
)


# ===================================================
# TEMPORAL COEFFICIENT OF VARIATION
# ===================================================

plot_long_term_map(
    raster_array=long_term_stats[
        "cv"
    ],

    title=(
        f"{LAKE_NAME}: "
        f"Temporal Variability in "
        f"{VIS_LABEL}"
    ),

    colorbar_label="CV (%)",

    cmap="viridis"
)

In [ ]:
#@title 7.9. Export Cross-Lake Visualization Data

VIS_SUMMARY_FILE = os.path.join(
    VIS_OUTPUT_DIR,
    (
        f"{VIS_DATA_SOURCE}_"
        f"{VIS_TARGET}_"
        f"{VIS_START_YEAR}-"
        f"{VIS_END_YEAR}_"
        f"Summary.xlsx"
    )
)


with pd.ExcelWriter(
    VIS_SUMMARY_FILE,
    engine="openpyxl"
) as writer:

    all_image_summary_df.to_excel(
        writer,
        sheet_name="Image_Statistics",
        index=False
    )

    all_monthly_summary_df.to_excel(
        writer,
        sheet_name="Monthly_Long_Format",
        index=False
    )

    monthly_mean_matrix.to_excel(
        writer,
        sheet_name="Monthly_Mean"
    )

    monthly_count_matrix.to_excel(
        writer,
        sheet_name="Image_Count"
    )


print(
    f"Visualization summary saved to:\n"
    f"{VIS_SUMMARY_FILE}"
)

---

### Author and Contact

This workflow was developed by **Rabia Munsaf Khan** as part of doctoral research at the
SUNY College of Environmental Science and Forestry (SUNY-ESF).

For questions, issues, or suggestions related to this workflow, please open an issue in the
GitHub repository or contact the author at:

**Email:** rkhan06@syr.edu; rabiamunsaf@gmail.com

If you use or adapt this workflow in your research, please cite the associated publication.
Citation details will be added upon publication.